# 🧠 Tu propio modelo experto: fine-tuning con QLoRA, self-hosting y un agente que lo usa

> ¿Le pagas a una API gigante para una tarea chica, repetitiva y siempre igual? Hay otra
> opción: entrenar un modelo chico, tuyo y experto en ESO, y hostearlo tú.

Hoy, paso a paso y en vivo, vamos a:

1. Ajustar (fine-tuning) un LLM chico con **QLoRA**, en la GPU T4 gratis de Colab.
2. **Servirlo nosotros mismos** con Ollama, sin depender de una API externa.
3. Construir un **agente** que lo use como herramienta, solo cuando conviene.

## Lo que te llevas

- Este notebook, que entrena un LLM y te deja un adapter de **megas, no de gigas**.
- Tu modelo experto, servido en tu propia máquina.
- Un agente que lo usa, y un criterio para saber **cuándo vale la pena** hacer fine-tuning.

| # | Sección | Minutos |
|---|---|---|
| 1 | El problema: por qué no mandar todo a una API gigante | 7 |
| 2 | Lab 1: QLoRA en vivo en una T4 | 15 |
| 3 | Lab 2: del adapter al modelo servido con Ollama | 13 |
| 4 | Lab 3: el agente usa nuestro modelo como herramienta | 15 |
| 5 | ¿Cuándo hacer fine-tuning y cuándo solo un buen prompt? | 6 |

Taller de Axel Sirota para Nerdearla México.

**Todas las celdas corren tal como están.** Los ejercicios son opcionales y no bloquean
nada: si te los saltas, el notebook sigue funcionando igual. Las soluciones están al final.

### 🎓 Data Trainers

Si lo que hacemos hoy te sirve, lo hacemos para empresas.

En [Data Trainers](https://www.datatrainersllc.com) armamos capacitaciones técnicas de IA
para equipos: agentes, LLMs, fine-tuning, MLOps y cloud. Igual que este taller, pero con el
stack de tu empresa, tus casos y tu gente, y con el tiempo suficiente para que quede.

Si tu equipo está empezando con esto y no sabe por dónde arrancar, escríbenos:
**[www.datatrainersllc.com](https://www.datatrainersllc.com)**

### 📱 Llévate el notebook

Escanea el código y se te abre este mismo notebook en tu Colab, para seguirlo en tu
computadora.

In [ ]:
# ===========================================================================
# QR PARA ABRIR ESTE NOTEBOOK (corre esta celda primero)
# ===========================================================================
# Va ANTES de instalar todo, así puedes escanear el QR mientras arranca el taller.
# Por eso instala su propia dependencia, que es chiquita.
!pip install -q "qrcode[pil]"

import qrcode
from qrcode.constants import ERROR_CORRECT_H


def hacer_qr(url: str, box: int = 6):
    """Arma un QR nítido y fácil de escanear de lejos.

    box_size dibuja cada módulo con ese tamaño en píxeles. Lo hacemos así en vez de
    agrandar la imagen después: si la reescalas, el suavizado borra los bordes y desde
    el fondo del salón no escanea. ERROR_CORRECT_H aguanta hasta un 30% del código tapado.
    """
    qr = qrcode.QRCode(error_correction=ERROR_CORRECT_H, box_size=box, border=3)
    qr.add_data(url)
    qr.make(fit=True)
    return qr.make_image(fill_color="black", back_color="white")


COLAB_URL = (
    "https://colab.research.google.com/github/"
    "axel-sirota/nerdearla_finetuning_agent/blob/main/finetuning_workshop.ipynb"
)

display(hacer_qr(COLAB_URL))
print(COLAB_URL)

## ⚙️ Sección 0: Preparamos el entorno

**Primero, la GPU.** En el menú de Colab: **Entorno de ejecución → Cambiar tipo de entorno
de ejecución → T4 GPU → Guardar.** Sin GPU, el fine-tuning no corre (la celda de abajo te
avisa si te olvidaste).

Después, dos celdas: la primera instala todo, la segunda te pide la clave de OpenAI.
Axel la comparte en el chat del taller. Cuando la pegues **no se ve en pantalla y no queda
guardada en el notebook**, así que puedes compartir el archivo después sin problema.

Si reinicias el entorno, te la vuelve a pedir. Es lo único que hay que repetir.

In [ ]:
# ===========================================================================
# INSTALAMOS TODO (PARA CASA: tarda un par de minutos la primera vez)
# ===========================================================================
# Versiones clavadas a propósito: son las que probamos juntas en una T4 real.
# OJO: NO reinstalamos torch. Colab ya trae uno compatible con su GPU, y el torch
# más nuevo de PyPI pide una versión de CUDA más nueva que la de Colab.
!pip install -q \
    transformers==5.23.0 \
    peft==0.21.3 \
    trl==1.18.1 \
    bitsandbytes==0.50.2 \
    datasets==5.2.0 \
    accelerate==1.16.0 \
    openai==3.54.0 \
    langchain==1.5.0 \
    langchain-core==1.8.2 \
    langchain-openai==1.8.1 \
    langgraph==1.2.22

import torch

print(f"Listo. torch {torch.__version__} (el de Colab, a propósito).")
print("Si no viste ningún error rojo, seguimos.")

In [ ]:
# ===========================================================================
# IMPORTS, GPU Y CONSTANTES DEL TALLER (NARRAR solo la clave y la GPU)
# ===========================================================================
import base64
import getpass
import json
import os

import requests
import torch
from IPython.display import Image, display
from openai import OpenAI

# --- La GPU --------------------------------------------------------------------
# Sin GPU no hay fine-tuning. Preferimos fallar aquí, claro y en español, que
# dentro de cinco minutos con un error de 40 líneas.
if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU. En Colab: Entorno de ejecución > Cambiar tipo de entorno de "
        "ejecución > T4 GPU > Guardar. Después vuelve a correr las celdas de arriba."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# OJO: en una T4, torch.cuda.is_bf16_supported() dice True (lo emula), pero la T4
# no tiene bfloat16 de verdad. Por eso en todo el taller usamos float16 a mano.

# --- Interruptor del taller ------------------------------------------------------
# True: no entrena en vivo, baja del Hub el adapter que ya entrenamos con este mismo
# código. Úsalo si no tienes tiempo o si Colab no te dio una T4.
USAR_ADAPTER_PREENTRENADO = False

# --- Los modelos y dónde vive cada cosa --------------------------------------------
MODEL_BASE = "Qwen/Qwen2.5-1.5B-Instruct"     # Hugging Face, para entrenar
MODEL_GPT = "gpt-4o-mini"                     # el modelo grande, por API
ADAPTER_REPO = "axelsirota/qwen2.5-1.5b-reclamos-conectamx-lora"   # respaldo en el Hub
ADAPTER_DIR = "adapter_reclamos"              # donde el Lab 1 guarda el adapter

REPO_RAW = ("https://raw.githubusercontent.com/"
            "axel-sirota/nerdearla_finetuning_agent/main")
DATA_URL = f"{REPO_RAW}/data/reclamos.csv"
CLIENTES_URL = f"{REPO_RAW}/data/clientes.csv"
DIAGRAMAS = f"{REPO_RAW}/diagrams"

# --- La clave ------------------------------------------------------------------------
# getpass no muestra lo que escribes: no queda en el notebook ni en la grabación.
# Es la ÚNICA clave del taller. Hugging Face no la necesita: el modelo es abierto.
OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY (no se va a ver): ")
client = OpenAI(api_key=OPENAI_API_KEY)
# LangChain, en el Lab 3, la busca en el entorno por su cuenta.
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


# --- Diagramas (PARA CASA) -------------------------------------------------------------
def mostrar_diagrama(slug: str) -> None:
    """Trae un diagrama Mermaid del repo y lo dibuja como imagen (Colab no dibuja Mermaid)."""
    fuente = requests.get(f"{DIAGRAMAS}/{slug}.mmd", timeout=30)
    if fuente.status_code != 200:
        print(f"(El diagrama '{slug}' todavía no está publicado.)")
        return
    codificado = base64.urlsafe_b64encode(fuente.text.encode()).decode()
    imagen = requests.get(f"https://mermaid.ink/img/{codificado}", timeout=30)
    if imagen.status_code != 200:
        print(f"(No se pudo dibujar '{slug}' ahora. El texto del diagrama está en el repo.)")
        return
    display(Image(data=imagen.content))


print(f"\nTodo listo. Modelo base: {MODEL_BASE}. Modelo grande: {MODEL_GPT}.")

## 🧩 Sección 1: El problema, ¿por qué no mandar todo a una API gigante?

Imagina que trabajas en **ConectaMX**, una compañía de internet, celular y TV en México.
Cada día llegan unos **10,000 reclamos por escrito**. Cada uno tiene que salir clasificado
como un JSON, para que el sistema abra el ticket correcto:

`{"categoria": "...", "urgencia": "...", "producto": "..."}`

Es una tarea chica, repetitiva y siempre igual. La pregunta de esta sección es simple:
**¿quién la hace?**

1. Un modelo chico y abierto, corriendo gratis en tu propia GPU.
2. Un modelo grande por API, como `gpt-4o-mini`.

Antes de decidir, vamos a ver los datos. En tu empresa, esto es tu historial de tickets.

In [ ]:
# ===========================================================================
# EL HISTORIAL DE RECLAMOS (NARRAR)
# ===========================================================================
import pandas as pd

# Un CSV en GitHub. En tu empresa, esto sale de tu sistema de tickets.
# Son unos 600 reclamos sintéticos de ConectaMX: un modelo grande escribió el texto
# para etiquetas que elegimos nosotros, y otro modelo los revisó a ciegas. Solo
# quedaron los reclamos donde los dos coincidieron en las tres etiquetas.
df = pd.read_csv(DATA_URL)

# La columna "split" ya viene armada: unos 500 para entrenar, 100 para medir.
# Los 100 de prueba NUNCA se usan para entrenar. Si los usáramos, el examen
# tendría las respuestas adentro.
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_test = df[df["split"] == "test"].reset_index(drop=True)

print(f"Reclamos totales: {len(df)}")
print(f"  para entrenar: {len(df_train)}")
print(f"  para medir:    {len(df_test)}")
print()

# Tres filas para ver la forma: el texto tal como lo escribió el cliente,
# y las tres etiquetas que queremos que el modelo devuelva.
df_train[["texto", "categoria", "urgencia", "producto"]].head(3)

In [ ]:
# ===========================================================================
# NUESTRAS ETIQUETAS Y EL PROMPT DEL CLASIFICADOR (NARRAR)
# ===========================================================================
# Estas listas son las reglas de ConectaMX. Ningún modelo del mundo las conoce:
# las inventamos nosotros. Fíjate en esto, porque es el centro de todo el taller.
CATEGORIAS = ["facturacion", "falla_servicio", "velocidad", "cancelacion", "atencion", "equipo"]
URGENCIAS = ["alta", "media", "baja"]
PRODUCTOS = ["fibra", "movil", "tv", "paquete"]

# UN solo prompt de sistema para todo el taller: para medir el modelo base,
# para entrenar, para servirlo en Ollama y para la tool del agente.
# Si cambias una coma aquí y no allá, el fine-tuning se rompe sin avisar.
SYSTEM_CLASIFICADOR = (
    "Eres el clasificador de reclamos de ConectaMX. "
    "Responde SOLO con un JSON con las claves categoria, urgencia y producto. "
    "categoria: facturacion, falla_servicio, velocidad, cancelacion, atencion o equipo. "
    "urgencia: alta, media o baja. "
    "producto: fibra, movil, tv o paquete."
)


def a_objetivo(fila) -> str:
    """El JSON que queremos que el modelo responda para un reclamo."""
    # Lo armamos desde las columnas del CSV, nunca lo guardamos aparte:
    # así el CSV y lo que aprende el modelo no pueden desincronizarse.
    return json.dumps({"categoria": fila["categoria"], "urgencia": fila["urgencia"],
                       "producto": fila["producto"]}, ensure_ascii=False)


def a_mensajes(fila) -> list[dict]:
    """Un reclamo en formato chat: system, user (el texto), assistant (el JSON)."""
    # Es el formato que usan todos los modelos de chat, y el que vamos a usar para
    # entrenar en el Lab 1. Tres turnos: las reglas, el reclamo y la respuesta correcta.
    return [
        {"role": "system", "content": SYSTEM_CLASIFICADOR},
        {"role": "user", "content": fila["texto"]},
        {"role": "assistant", "content": a_objetivo(fila)},
    ]


# Así se ve UN ejemplo de entrenamiento. Unos 500 de estos son todo lo que hace falta.
for mensaje in a_mensajes(df_train.iloc[0]):
    print(f"[{mensaje['role']}] {mensaje['content']}\n")

In [ ]:
# ===========================================================================
# LA REGLA PARA MEDIR (PARA CASA: no la leemos en vivo, pero úsala en tu empresa)
# ===========================================================================
import re
import time


def parsear_json(texto: str) -> dict | None:
    """Intenta leer el JSON de una respuesta. None si no es JSON válido con las 3 claves."""
    # Los modelos chicos suelen envolver el JSON en ```json ... ``` o agregar una frase
    # antes. Eso lo limpia cualquier código en una línea, así que NO lo contamos como
    # error: buscamos el primer {...} y lo leemos. Lo que sí es error es que no haya
    # JSON o que falte una clave.
    if not isinstance(texto, str):
        return None
    bloque = re.search(r"\{.*\}", texto, re.DOTALL)
    if bloque is None:
        return None
    try:
        datos = json.loads(bloque.group(0))
    except json.JSONDecodeError:
        return None
    if not isinstance(datos, dict):
        return None
    if not all(clave in datos for clave in ("categoria", "urgencia", "producto")):
        return None
    return datos


def evaluar(predecir, df) -> dict:
    """Corre predecir(texto) -> str sobre cada fila y mide.

    Devuelve {"json_valido": %, "categoria": %, "urgencia": %, "producto": %,
              "exacto": %, "segundos_por_reclamo": float}.
    """
    # predecir es cualquier función que reciba un texto y devuelva un string.
    # Así medimos con la MISMA regla al modelo base, al modelo entrenado, al modelo
    # servido en Ollama y a gpt-4o-mini. La misma vara para todos.
    aciertos = {"json_valido": 0, "categoria": 0, "urgencia": 0, "producto": 0, "exacto": 0}
    inicio = time.time()
    for i, fila in enumerate(df.itertuples(index=False), start=1):
        datos = parsear_json(predecir(fila.texto))
        if datos is not None:
            aciertos["json_valido"] += 1
            bien = [datos.get(clave) == getattr(fila, clave)
                    for clave in ("categoria", "urgencia", "producto")]
            aciertos["categoria"] += bien[0]
            aciertos["urgencia"] += bien[1]
            aciertos["producto"] += bien[2]
            aciertos["exacto"] += all(bien)
        # Imprimimos el avance para que la pantalla nunca se vea congelada.
        if i % 20 == 0:
            print(f"  {i}/{len(df)} reclamos")
    total = len(df)
    resultado = {clave: round(100 * valor / total, 1) for clave, valor in aciertos.items()}
    resultado["segundos_por_reclamo"] = round((time.time() - inicio) / total, 2)
    return resultado


print("Listo: parsear_json y evaluar definidas.")

In [ ]:
# ===========================================================================
# MUESTRA: los mismos 5 reclamos para el antes y el después (NARRAR)
# ===========================================================================
# Elegimos 5 reclamos de PRUEBA, a mano, porque cada uno pone a prueba una regla
# distinta de ConectaMX: un router que no sirve (eso es "equipo", no
# "falla_servicio"), un celular con internet lento, un modem que falla un poco y el
# cliente lo cuenta con calma (urgencia "baja"), alguien que se queja de la
# atención y un cobro doble. Los vamos a usar aquí, después de entrenar y cuando el
# modelo esté servido. Siempre los mismos.
IDS_MUESTRA = [183, 117, 107, 24, 41]
_filas_muestra = df_test.set_index("id").loc[IDS_MUESTRA]
MUESTRA = _filas_muestra["texto"].tolist()

for numero, (texto, (_, fila)) in enumerate(zip(MUESTRA, _filas_muestra.iterrows()), start=1):
    print(f"{numero}. {texto}")
    print(f"   lo correcto: {a_objetivo(fila)}\n")

### 🗺️ Dos caminos para el mismo reclamo

<!-- DIAGRAM: api-vs-propio. graph LR. Arriba: cada reclamo sale de ConectaMX por internet hacia una API paga (gpt-4o-mini), con etiquetas "pago por llamada" y "los datos salen". Abajo: el reclamo se queda adentro, va a un modelo chico en nuestra GPU, etiquetas "gratis" y "privado". Ambos terminan en el mismo JSON. Max 8 nodos, ASCII. -->

In [ ]:
# Dos caminos, el mismo JSON al final. La pregunta es cuál lo llena bien.
mostrar_diagrama("api-vs-propio")

Arriba, el camino de siempre: cada reclamo viaja a un servidor que no es tuyo, y pagas por
cada uno. Abajo, el camino que vamos a construir hoy: el reclamo nunca sale de tu GPU.

Los dos terminan en el mismo JSON. La pregunta es cuál lo llena **bien**, con las reglas de
ConectaMX. Vamos a probar los dos con los mismos 5 reclamos.

### 🖥️ Opción 1: un modelo chico, gratis, en tu GPU

Vamos a usar **Qwen2.5-1.5B-Instruct**: 1,500 millones de parámetros, licencia Apache 2.0,
sin registro ni token. Es un modelo de chat general. No sabe nada de ConectaMX.

Lo cargamos **en 4 bits**. Eso es la **cuantización**: guardar cada peso con 4 bits en vez
de 16, como bajar la resolución de una foto. El modelo pasa de unos 3 GB a algo más de 1 GB
en la GPU, y casi no pierde calidad. Es el mismo truco que, en el Lab 1, nos deja entrenarlo
en una T4 gratis.

Mientras carga, fíjate en el número de memoria que imprime al final.

In [ ]:
# ===========================================================================
# EL MODELO BASE EN 4 BITS (NARRAR mientras carga: tarda entre 15 s y 1 minuto)
# ===========================================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# La receta de la cuantización. Cuatro decisiones, todas pensadas para una T4:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # cada peso ocupa 4 bits en vez de 16
    bnb_4bit_quant_type="nf4",             # NF4: un formato de 4 bits pensado para
                                           # pesos de redes neuronales (el de QLoRA)
    bnb_4bit_compute_dtype=torch.float16,  # las cuentas se hacen en float16. La T4
                                           # NO tiene bfloat16 de verdad, aunque
                                           # torch diga que sí: no le creas
    bnb_4bit_use_double_quant=True,        # cuantiza también las constantes de la
                                           # cuantización: ahorra un poco más
)

# El tokenizer convierte texto en números (tokens) y de vuelta.
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

# El modelo. Lo cargamos UNA sola vez en todo el notebook: el Lab 1 entrena este
# mismo objeto. Cargarlo dos veces en una T4 es pedir un error de memoria.
# Si tarda mucho en tu casa: crea un token en huggingface.co y guárdalo como
# secreto HF_TOKEN en Colab (la llave a la izquierda). Con token, las descargas
# tienen límites más altos.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    quantization_config=bnb_config,
    device_map="auto",       # que transformers lo ponga en la GPU solo
    dtype=torch.float16,     # el resto de las capas (las que no van en 4 bits)
)

print(f"Modelo cargado: {MODEL_BASE}")
print(f"Memoria que ocupa en la GPU: {model.get_memory_footprint() / 1e9:.2f} GB "
      f"de los ~15 GB de la T4")

In [ ]:
# ===========================================================================
# generar(texto, modelo): le pasamos un reclamo y nos devuelve su respuesta (NARRAR)
# ===========================================================================
def generar(texto: str, modelo) -> str:
    """Le manda un reclamo al modelo local con nuestro prompt y devuelve su respuesta."""
    # Recibe el modelo como parámetro a propósito: en el Lab 1 vamos a llamar a esta
    # MISMA función con el modelo ya entrenado, y comparar.
    mensajes = [
        {"role": "system", "content": SYSTEM_CLASIFICADOR},
        {"role": "user", "content": texto},
    ]
    # apply_chat_template arma el texto con las marcas especiales de Qwen
    # (<|im_start|>system ...). add_generation_prompt=True deja abierto el turno del
    # asistente, para que el modelo escriba la respuesta.
    entrada = tokenizer.apply_chat_template(
        mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(modelo.device)
    with torch.no_grad():  # solo inferencia: no guardamos gradientes, ahorra memoria
        salida = modelo.generate(
            **entrada,
            max_new_tokens=60,   # un JSON de 3 claves entra de sobra en 60 tokens
            do_sample=False,     # siempre la opción más probable: mismo reclamo,
                                 # misma respuesta. Para clasificar no queremos creatividad
            pad_token_id=tokenizer.eos_token_id,
        )
    # generate devuelve el prompt + la respuesta. Cortamos el prompt.
    nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(nuevos, skip_special_tokens=True).strip()


print("Listo: generar(texto, modelo) definida.")

In [ ]:
# ===========================================================================
# EL MODELO BASE CONTRA NUESTROS 5 RECLAMOS (NARRAR, el primer dolor)
# ===========================================================================
# Lee la respuesta CRUDA de cada uno, y después cada etiqueta contra lo correcto.
_correctas = df_test.set_index("id").loc[IDS_MUESTRA]
_listas = {"categoria": CATEGORIAS, "urgencia": URGENCIAS, "producto": PRODUCTOS}

for numero, (texto, (_, fila)) in enumerate(zip(MUESTRA, _correctas.iterrows()), start=1):
    respuesta = generar(texto, model)
    print(f"--- Reclamo {numero}: {texto[:80]}...")
    print(f"Respuesta cruda: {respuesta!r}")
    datos = parsear_json(respuesta)
    if datos is None:
        print("❌ No hay un JSON con las 3 claves\n")
        continue
    for clave, valores_validos in _listas.items():
        dijo, correcto = datos.get(clave), fila[clave]
        if dijo == correcto:
            marca = "✅"
        elif dijo not in valores_validos:
            marca = "❌ (ese valor ni siquiera está en nuestra lista)"
        else:
            marca = f"❌ (lo correcto es {correcto})"
        print(f"  {clave:10} = {dijo!s:15} {marca}")
    print()

Fíjate en tres cosas:

1. **El JSON casi siempre sale.** Viene envuelto en ```json, pero eso se limpia con una línea
   de código. El formato no es el problema grave.
2. **Las etiquetas están mal.** Casi todo es urgencia "alta". Un router que no sirve termina
   como "falla_servicio". A veces inventa valores que no existen en nuestra lista.
3. **No es que el modelo sea tonto.** Nadie le dijo que en ConectaMX un modem es "equipo".
   Esas reglas son nuestras.

En los 100 reclamos de prueba, este modelo acierta la categoría el **36%** de las veces y
las tres etiquetas juntas solo el **7%** <!-- NUMERO v3 --> (lo medimos antes del taller en
una T4, y lo vamos a volver a medir en vivo en el Lab 1).

### ☁️ Opción 2: la API grande

Ahora el mismo reclamo a `gpt-4o-mini`. Y para que la pelea sea justa, le damos todas las
ventajas: usamos **Structured Outputs**, que obliga al modelo a responder con un JSON que
cumple un esquema. Le pasamos nuestras tres listas como las únicas opciones válidas. Así, el
formato queda perfecto por construcción y solo medimos lo que importa: **¿conoce nuestras
etiquetas?**

In [ ]:
# ===========================================================================
# predecir_gpt(texto): el mismo reclamo, por API, con formato garantizado (NARRAR)
# ===========================================================================
from typing import Literal

from pydantic import BaseModel


class Clasificacion(BaseModel):
    """El esquema que gpt-4o-mini está OBLIGADO a cumplir."""
    # Literal con nuestras listas: la API solo puede elegir uno de estos valores.
    # Con esto, inventar "n/a" o poner "equipo" en producto es imposible.
    categoria: Literal[tuple(CATEGORIAS)]
    urgencia: Literal[tuple(URGENCIAS)]
    producto: Literal[tuple(PRODUCTOS)]


# Contamos los tokens de cada llamada: la API cobra por token, y en la Sección 5 vamos
# a proyectar el costo a 10,000 reclamos por día.
USO_GPT = {"llamadas": 0, "tokens_entrada": 0, "tokens_salida": 0}


def predecir_gpt(texto: str) -> str:
    """Clasifica un reclamo con gpt-4o-mini y devuelve el JSON como texto."""
    respuesta = client.chat.completions.parse(
        model=MODEL_GPT,
        temperature=0,                   # misma idea que do_sample=False
        response_format=Clasificacion,   # Structured Outputs: el JSON cumple el esquema
        messages=[
            {"role": "system", "content": SYSTEM_CLASIFICADOR},
            {"role": "user", "content": texto},
        ],
    )
    USO_GPT["llamadas"] += 1
    USO_GPT["tokens_entrada"] += respuesta.usage.prompt_tokens
    USO_GPT["tokens_salida"] += respuesta.usage.completion_tokens
    # Devolvemos un string, igual que generar(): así evaluar() mide a los dos con la
    # misma regla, sin saber quién respondió.
    return json.dumps(respuesta.choices[0].message.parsed.model_dump(), ensure_ascii=False)


print("Listo: predecir_gpt(texto) definida.")

In [ ]:
# ===========================================================================
# gpt-4o-mini CONTRA LOS MISMOS 5 RECLAMOS, Y LO QUE CUESTA (NARRAR, el segundo dolor)
# ===========================================================================
_correctas = df_test.set_index("id").loc[IDS_MUESTRA]

for numero, (texto, (_, fila)) in enumerate(zip(MUESTRA, _correctas.iterrows()), start=1):
    datos = parsear_json(predecir_gpt(texto))
    print(f"--- Reclamo {numero}: {texto[:80]}...")
    for clave in ("categoria", "urgencia", "producto"):
        dijo, correcto = datos[clave], fila[clave]
        marca = "✅" if dijo == correcto else f"❌ (lo correcto es {correcto})"
        print(f"  {clave:10} = {dijo:15} {marca}")
    print()

# ¿Cuánto costaría a escala? Precio público de gpt-4o-mini (noviembre 2026):
# USD 0.15 por millón de tokens de entrada y USD 0.60 por millón de salida.
PRECIO_ENTRADA_USD = 0.15 / 1_000_000
PRECIO_SALIDA_USD = 0.60 / 1_000_000
costo_por_reclamo = (USO_GPT["tokens_entrada"] * PRECIO_ENTRADA_USD
                     + USO_GPT["tokens_salida"] * PRECIO_SALIDA_USD) / USO_GPT["llamadas"]

print(f"Tokens por reclamo: {USO_GPT['tokens_entrada'] / USO_GPT['llamadas']:.0f} de entrada, "
      f"{USO_GPT['tokens_salida'] / USO_GPT['llamadas']:.0f} de salida")
print(f"10,000 reclamos por día: USD {costo_por_reclamo * 10_000:.2f} al día, "
      f"USD {costo_por_reclamo * 10_000 * 30:.2f} al mes")
print("Y cada uno de esos 10,000 reclamos salió de ConectaMX hacia un servidor externo.")

### 📌 Lo que acabamos de ver

<!-- NUMERO v3: medido en una T4 y con la API sobre los 100 reclamos de prueba -->

| | Modelo chico (base) | gpt-4o-mini con esquema |
|---|---|---|
| ¿Dónde corre? | En tu GPU | En un servidor externo |
| ¿Cuánto cuesta? | Gratis | Por cada reclamo |
| ¿Los datos salen? | No | Sí, cada reclamo |
| JSON válido (100 de prueba) | 100% (con limpieza) | 100% |
| Categoría correcta | 36% | 79% |
| Urgencia correcta | 39% | 52% |
| Las 3 etiquetas bien | 7% | 39% |

Seamos honestos con los números:

- Con `gpt-4o-mini` el costo es bajo: unos dólares al mes a este volumen. El problema más
  grande es otro: **cada reclamo sale de tu empresa**, y aun así **tampoco conoce tus
  reglas**. Acierta la urgencia la mitad de las veces: casi todo lo marca "alta".
- El modelo chico es gratis y privado, pero hoy no sirve: no conoce nuestras etiquetas.

Lo que le falta al modelo chico no es inteligencia, son **tus ejemplos**. Tienes unos 500.
En el Lab 1 se los vamos a enseñar, en esta misma T4, en unos minutos.

In [ ]:
# EJERCICIO 1 (opcional, para después)
# ¿Y si solo mejoramos el prompt? Copia SYSTEM_CLASIFICADOR en una variable nueva
# (por ejemplo prompt_con_ejemplo) y agrégale una regla y un ejemplo:
#   'Un modem o router que falla es "equipo". Ejemplo: "mi router se apaga solo" ->
#    {"categoria": "equipo", "urgencia": "media", "producto": "fibra"}'
# Después arma la lista de mensajes a mano con ese prompt y llama a model.generate
# con MUESTRA[0] (el del router). ¿Mejora? ¿Y con los otros 4?
# Pista: copia el cuerpo de generar() y cambia solo el contenido del mensaje system.
# No recargues el modelo: usa el mismo `model`.
# Solución al final del notebook.

# EJERCICIO 2 (opcional, para después)
# El modelo base dijo urgencia "alta" para casi todo. ¿Qué tan seguido es "alta" de
# verdad? Cuenta los valores de la columna urgencia en df_train y compáralos.
# Pista: df_train["urgencia"].value_counts(normalize=True)
# Solución al final del notebook.

## 🔧 Sección 2 (Lab 1): QLoRA en vivo en una T4

En la sección anterior viste al modelo base fallar: envuelve el JSON en bloques de markdown,
se equivoca de categoría y dice "alta" a casi todo. Le explicamos la tarea en el prompt y no
alcanzó.

Ahora le vamos a enseñar la tarea con ejemplos. Unos 500 reclamos etiquetados, una GPU
gratis y unos minutos. Al final vas a tener un archivo de unos 74 MB que convierte a este
modelo en un experto en los reclamos de ConectaMX.

Primero medimos dónde estamos parados, con los mismos 100 reclamos de prueba de siempre.

In [ ]:
# ===========================================================================
# LA LÍNEA BASE: EL MODELO SIN ENTRENAR (NARRAR)
# ===========================================================================
# Corremos el modelo base sobre los 100 reclamos de prueba y medimos.
# Tarda unos 2 minutos y medio en una T4 (1.5 segundos por reclamo).
# Mientras corre, sigue leyendo la celda de abajo: Jupyter espera solo.
#
# evaluar() viene de la sección 1. Recibe una función texto -> respuesta y una tabla,
# y devuelve porcentajes por campo. La usamos igual antes y después de entrenar, así
# la comparación es justa.
inicio = time.time()
metricas_base = evaluar(lambda texto: generar(texto, model), df_test)

print(f"Listo en {time.time() - inicio:.0f} segundos.\n")
print("Modelo base, sobre 100 reclamos de prueba:")
for campo, valor in metricas_base.items():
    print(f"  {campo:22s} {valor}")

### ¿Por qué no entrenar el modelo entero?

Entrenar un modelo "a la antigua" significa actualizar todos sus pesos. Para cada peso
necesitas guardar el peso, su gradiente y dos números más del optimizador Adam. En la
práctica son unos 16 bytes por parámetro.

Nuestro modelo tiene 1,543 millones de parámetros. Hagamos la cuenta en la celda de abajo.

In [ ]:
# ===========================================================================
# LA CUENTA QUE ASUSTA (NARRAR)
# ===========================================================================
# Parámetros de Qwen2.5-1.5B-Instruct, contados con PEFT sobre la arquitectura real.
PARAMETROS_BASE = 1_543_714_304

# Fine-tuning completo con Adam en precisión mixta, por cada parámetro:
#   2 bytes  peso en fp16
#   2 bytes  gradiente en fp16
#   4 bytes  copia maestra del peso en fp32
#   8 bytes  dos momentos de Adam en fp32
BYTES_POR_PARAMETRO = 2 + 2 + 4 + 8

gb_completo = PARAMETROS_BASE * BYTES_POR_PARAMETRO / 1e9
gb_t4 = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Fine-tuning completo necesita: ~{gb_completo:.1f} GB (sin contar activaciones)")
print(f"Tu GPU gratis tiene:            {gb_t4:.1f} GB")
print(f"Uso actual de la GPU:           {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("\nNo entra. Ni de cerca. Hace falta otro camino.")

### LoRA: no tocas el modelo, le agregas un parche

**LoRA congela todos los pesos del modelo y entrena, al lado de cada capa, dos matrices
chiquitas (A y B) cuyo producto es la corrección que la capa necesita.**

El modelo original no cambia. Lo que aprende vive aparte, en un archivo pequeño: el adapter
(adaptador).

<!-- DIAGRAM: lora. graph TD. Arriba una entrada x. Dos caminos en paralelo: a la izquierda una caja grande W congelado 1536 x 1536; a la derecha dos cajas chicas en serie, A 1536 x 16 y luego B 16 x 1536, marcadas como entrenables. Los dos caminos se suman y salen a la salida h. Nota: solo A y B se entrenan, 1.18 por ciento de los parametros. ASCII, maximo 8 nodos. -->

In [ ]:
# El peso grande W queda congelado. Solo aprenden A y B, que son chiquitas.
mostrar_diagrama("lora")

Lee el diagrama así: la entrada pasa por el peso original, que no cambia, y también por el
atajo A y B. Las dos salidas se suman. Como A y B tienen un lado de tamaño 16 (el "rango",
`r`), entrenan una fracción mínima de lo que entrenaría el modelo completo.

### QLoRA: lo mismo, con el modelo base comprimido a 4 bits

**QLoRA es LoRA sobre un modelo base cuantizado a 4 bits: el modelo congelado ocupa la
cuarta parte de memoria y los adapters se entrenan en precisión normal encima de él.**

Ya cargaste el modelo en 4 bits en la sección 1, con `bnb_config`. Por eso hoy usa poco más
de 1 GB de la GPU en lugar de 3. Esa cuantización es la "Q" de QLoRA.

<!-- DIAGRAM: qlora. graph LR. A la izquierda una caja Qwen 1.5B base en 4 bits NF4, congelado, 1.1 GB. A la derecha, encima, una caja Adapter LoRA en fp32, entrenable, 74 MB. Flecha de la base al adapter con etiqueta se calcula en fp16. Abajo una barra GPU T4 15 GB con texto pico medido 2.7 GB. ASCII, maximo 6 nodos. -->

In [ ]:
# La base en 4 bits no se entrena. El adapter, en precisión completa, sí.
mostrar_diagrama("qlora")

La base comprimida solo se lee; nunca se actualiza. Los cálculos se hacen en fp16 y el
adapter guarda sus pesos en fp32 para no perder precisión al aprender. Resultado: todo el
entrenamiento entra en una T4 gratis con memoria de sobra.

### Paso 1: ponerle el adapter al modelo

Vamos a hacer tres cosas sobre el **mismo** `model` que ya está en la GPU. No lo volvemos a
cargar: en una T4, cargarlo dos veces es la forma más rápida de quedarte sin memoria.

1. `prepare_model_for_kbit_training`: deja el modelo de 4 bits listo para entrenar.
2. `LoraConfig`: dónde van los adapters y de qué tamaño.
3. `get_peft_model`: los pega al modelo y congela todo lo demás.

In [ ]:
# ===========================================================================
# EL ADAPTER LORA (NARRAR)
# ===========================================================================
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    # r es el rango: el lado chico de las matrices A y B. 16 es un valor estándar para
    # tareas acotadas como la nuestra. Más grande: más capacidad y más memoria.
    r=16,
    # lora_alpha escala la corrección de A y B. La regla práctica es 2 * r.
    lora_alpha=32,
    # Apaga al azar el 5% del adapter en cada paso, para no memorizar los ejemplos.
    lora_dropout=0.05,
    # En qué capas de Qwen2.5 ponemos adapters: las 7 proyecciones lineales de cada
    # bloque. Las 4 de atención (q, k, v, o) y las 3 del MLP (gate, up, down).
    # Estos nombres salen de la arquitectura del modelo; para otro modelo cambian.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    # Le dice a PEFT que es un modelo que genera texto.
    task_type="CAUSAL_LM",
)

if isinstance(model, PeftModel):
    # Si vuelves a correr esta celda, no le pegamos un segundo adapter encima.
    print("El modelo ya tiene su adapter. No hacemos nada.")
elif USAR_ADAPTER_PREENTRENADO:
    # Camino rápido: bajamos del Hub el adapter que entrenamos antes, con este mismo
    # código, y lo pegamos sobre la MISMA base en 4 bits. Pesa 74 MB.
    print(f"Bajando el adapter ya entrenado: {ADAPTER_REPO}")
    model = PeftModel.from_pretrained(model, ADAPTER_REPO)
else:
    # Prepara la base de 4 bits para entrenar: activa el gradient checkpointing
    # (recalcula en lugar de guardar, ahorra memoria) y pasa las capas de
    # normalización a fp32 para que el entrenamiento sea estable.
    model = prepare_model_for_kbit_training(model)
    # Pega los adapters y congela todo lo demás. Mismo objeto, ahora envuelto.
    model = get_peft_model(model, lora_config)

# La línea que vinimos a ver: cuánto vamos a entrenar de verdad.
model.print_trainable_parameters()

### Paso 2: los datos en formato de chat

El modelo aprende de pares **pregunta y respuesta**. La pregunta es el system prompt más el
reclamo. La respuesta es el JSON correcto.

Los separamos en `prompt` y `completion` por una razón: así el entrenamiento solo calcula el
error sobre la respuesta. No queremos que el modelo aprenda a escribir reclamos; queremos que
aprenda a contestarlos.

In [ ]:
# ===========================================================================
# LOS RECLAMOS EN FORMATO PROMPT Y COMPLETION (NARRAR)
# ===========================================================================
from datasets import Dataset


def a_prompt_completion(fila) -> dict:
    """Parte la conversación: system y user van al prompt, el JSON a la completion."""
    mensajes = a_mensajes(fila)          # [system, user, assistant], de la sección 1
    return {"prompt": mensajes[:2], "completion": mensajes[2:]}


train_dataset = Dataset.from_list([a_prompt_completion(fila) for _, fila in df_train.iterrows()])

print(f"Ejemplos de entrenamiento: {len(train_dataset)}\n")
ejemplo = train_dataset[0]
print("PROMPT (el modelo lo lee, no aprende a escribirlo):")
print("  user:", ejemplo["prompt"][1]["content"][:120], "...")
print("\nCOMPLETION (lo único sobre lo que se calcula el error):")
print("  assistant:", ejemplo["completion"][0]["content"])

In [ ]:
# ===========================================================================
# LA CONFIGURACIÓN DEL ENTRENAMIENTO (NARRAR)
# ===========================================================================
from trl import SFTConfig, SFTTrainer

# 120 pasos de 8 reclamos = casi 2 pasadas por los ejemplos de entrenamiento.
# En una T4 tarda unos 3 minutos (medido: 1.46 segundos por paso).
# ¿Vas atrasado? Bájalo a 60: tarda la mitad y aun así mejora mucho.
MAX_STEPS = 120

sft_config = SFTConfig(
    output_dir="outputs",              # checkpoints temporales; no los guardamos
    max_steps=MAX_STEPS,
    per_device_train_batch_size=8,     # 8 reclamos por paso
    learning_rate=2e-4,                # alto para fine-tuning completo, normal para LoRA
    lr_scheduler_type="cosine",        # baja el learning rate suave hacia el final
    warmup_steps=5,                    # arranca despacio los primeros 5 pasos
    logging_steps=10,                  # imprime la pérdida cada 10 pasos
    max_length=256,                    # el ejemplo más largo mide unos 220 tokens
    # LA TRAMPA DE LA T4: no soporta bf16 de verdad. TRL usa bf16 por defecto si no
    # le dices nada, así que lo apagamos y pedimos fp16 explícito.
    fp16=True,
    bf16=False,
    save_strategy="no",                # guardamos el adapter a mano, al final
    report_to="none",                  # sin wandb ni cuentas externas
    seed=42,
)

if USAR_ADAPTER_PREENTRENADO:
    trainer = None
    print("Modo adapter preentrenado: no armamos el trainer.")
else:
    trainer = SFTTrainer(
        model=model,                   # el MISMO model, ya con su adapter
        args=sft_config,
        train_dataset=train_dataset,
        processing_class=tokenizer,    # aplica el chat template de Qwen por nosotros
    )
    print("Trainer listo. Pérdida solo sobre la respuesta:", trainer.completion_only_loss)

In [ ]:
# ===========================================================================
# UN DETALLE QUE TE AHORRA UN ERROR EN LA T4 (PARA CASA)
# ===========================================================================
# Al armar el trainer, TRL convierte los adapters de un modelo de 4 bits a bf16.
# En una GPU moderna está bien. En una T4, el entrenamiento en fp16 revienta en el
# primer paso con:
#   NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda"
#   not implemented for 'BFloat16'
# Lo vimos pasar en una T4 real. La solución: pasar los adapters a fp32.
# Son solo 18 millones de números: cuesta 74 MB y nada de velocidad.
if trainer is not None:
    for parametro in model.parameters():
        if parametro.requires_grad:
            parametro.data = parametro.data.float()

tipos = {str(p.dtype) for p in model.parameters() if p.requires_grad}
print("Tipo de los pesos entrenables:", tipos or "ninguno (modo preentrenado)")

In [ ]:
# ===========================================================================
# A ENTRENAR (NARRAR)
# ===========================================================================
# Mira la columna "Training Loss" en la tabla que aparece abajo: es el error del
# modelo sobre el JSON. En nuestra corrida en una T4 bajó de unos 0.3 en el paso 10
# a unos 0.05 al final. Si baja, el modelo está aprendiendo.
# Memoria pico medida: unos 2.7 GB de los 15 de la T4.
if trainer is not None:
    inicio = time.time()
    trainer.train()
    segundos = time.time() - inicio
    print(f"\nEntrenamiento listo: {MAX_STEPS} pasos en {segundos:.0f} segundos "
          f"({segundos / MAX_STEPS:.2f} s por paso).")
    print(f"Memoria pico de la GPU: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
else:
    print("Modo adapter preentrenado: nos saltamos el entrenamiento.")

In [ ]:
# ===========================================================================
# EL ADAPTER: TODO LO QUE APRENDIÓ, EN UNA CARPETA (NARRAR)
# ===========================================================================
# save_pretrained de un modelo PEFT guarda SOLO el adapter, no la base.
# Funciona igual si lo entrenaste o si lo bajaste del Hub.
model.save_pretrained(ADAPTER_DIR)

total = 0
for archivo in sorted(os.listdir(ADAPTER_DIR)):
    tamano = os.path.getsize(os.path.join(ADAPTER_DIR, archivo))
    total += tamano
    print(f"  {archivo:30s} {tamano / 1e6:8.2f} MB")

print(f"\nAdapter completo: {total / 1e6:.0f} MB")
print("El modelo base en fp16 pesa unos 3,100 MB. El adapter es menos del 3% de eso.")

### El momento de la verdad

Mismo objeto `model`, mismos reclamos. Con `disable_adapter()` apagamos el adapter un
momento y vemos al modelo base; sin eso, vemos a nuestro modelo. No cargamos nada nuevo.

In [ ]:
# ===========================================================================
# ANTES Y DESPUÉS, LADO A LADO (NARRAR)
# ===========================================================================
model.eval()   # modo inferencia: apaga el dropout del adapter

for i, texto in enumerate(MUESTRA, start=1):
    with model.disable_adapter():           # adapter apagado: el modelo base
        antes = generar(texto, model)
    despues = generar(texto, model)         # adapter prendido: nuestro modelo
    print(f"[{i}] {texto[:90]}...")
    print(f"    ANTES:   {antes[:110]!r}")
    print(f"    DESPUÉS: {despues}")
    print(f"    JSON válido: {parsear_json(despues) is not None}\n")

In [ ]:
# ===========================================================================
# LA MEDICIÓN QUE IMPORTA (NARRAR)
# ===========================================================================
# Los mismos 100 reclamos, la misma función evaluar(). Tarda unos 3 minutos y medio:
# con el adapter sin fusionar, cada respuesta es un poco más lenta. En la sección 3 lo
# servimos con Ollama y eso cambia.
inicio = time.time()
metricas_ft = evaluar(lambda texto: generar(texto, model), df_test)
print(f"Listo en {time.time() - inicio:.0f} segundos.\n")

antes_despues = pd.DataFrame({"modelo base": metricas_base, "nuestro modelo": metricas_ft})
print(antes_despues.round(2).to_string())

### Qué acabamos de ver

<!-- NUMERO v3: en nuestra corrida en una T4, exacto pasó de 7 a 65 con 120 pasos (71 con el adapter del Hub); gpt-4o-mini, 39 -->

Con unos 3 minutos de una GPU gratis, el modelo pasó de acertar la categoría 4 de cada 10
veces a acertarla casi siempre. El producto, igual. Y responde JSON limpio, sin adornos.

Fíjate en la urgencia, que es la más difícil: el modelo base decía "alta" a casi todo, y
`gpt-4o-mini` también. El nuestro aprendió la regla de ConectaMX: en nuestra corrida casi
nunca confunde "alta" con "baja", y los errores que le quedan son de un solo paso ("alta"
contra "media"). Decidir si algo es urgente es discutible hasta para dos personas: **la
calidad de tus etiquetas es el techo de tu modelo.**

Y en las tres etiquetas juntas, este modelo de 1.5B entrenado 3 minutos le gana a
`gpt-4o-mini`: en nuestra corrida, 65% contra 39%.

El fine-tuning le enseñó **formato y comportamiento**: qué etiquetas existen y cómo
responder. No le enseñó datos nuevos del mundo. Para eso existe RAG.

Y todo eso vive en una carpeta de 74 MB. Un archivo no atiende clientes, así que en la
próxima sección lo vamos a servir.

In [ ]:
# EJERCICIO 3 (opcional, para después)
# ¿Cuántos parámetros entrenaría LoRA con r=8 en lugar de r=16?
# No vuelvas a entrenar ni a cargar nada: es pura aritmética.
# Pista: cada capa lineal de tamaño (entrada, salida) suma r * (entrada + salida)
# parámetros de adapter. En Qwen2.5-1.5B, por bloque:
#   q_proj 1536 -> 1536    k_proj 1536 -> 256    v_proj 1536 -> 256
#   o_proj 1536 -> 1536    gate_proj 1536 -> 8960    up_proj 1536 -> 8960
#   down_proj 8960 -> 1536
# y hay 28 bloques. Comprueba primero que con r=16 te da 18,464,768.
# Solución al final del notebook.

# EJERCICIO 4 (opcional, para después)
# Busca los reclamos de prueba donde nuestro modelo se equivocó en la urgencia y
# léelos. ¿Tú habrías elegido la etiqueta original o la del modelo?
# Usa solo inferencia con el model que ya tienes: nada de reentrenar.
# Limítate a los primeros 30 reclamos de df_test para que tarde un minuto.
# Solución al final del notebook.

## 🏠 Sección 3 (Lab 2): Self-hosting, del adapter al modelo servido

Al final del Lab 1 nos quedó un adapter de unos 70 MB en la carpeta `adapter_reclamos`.
Clasifica bien los reclamos, pero hay un problema: **una carpeta no contesta preguntas.**

Tu app, tu agente o tu sistema de tickets no pueden "llamar" a una carpeta. Necesitan una
URL, un servicio que esté escuchando y que responda en un formato conocido.

En este lab vamos a convertir esa carpeta en un servicio:

1. Instalamos **Ollama** dentro de esta misma máquina de Colab.
2. Le cargamos el modelo base y, encima, **nuestro adapter**.
3. Lo llamamos con el mismo cliente de OpenAI que ya conoces, cambiando solo la URL.

Todo en la misma T4 gratis. Ningún reclamo sale de esta máquina.

In [ ]:
# ===========================================================================
# EL PROBLEMA: tenemos un adapter, pero nadie lo sirve (NARRAR)
# ===========================================================================
import urllib.request

from huggingface_hub import snapshot_download

# PARA CASA: si reiniciaste el entorno, la carpeta puede no existir. En ese caso bajamos
# el adapter ya entrenado del Hub. Si la carpeta existe, esta línea no hace nada.
if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")):
    snapshot_download(ADAPTER_REPO, local_dir=ADAPTER_DIR)

# Miremos qué hay en la carpeta. Fíjate en el tamaño: decenas de MB, no GB.
# Es solo la "diferencia" que aprendió el modelo, no el modelo entero.
print(f"Contenido de {ADAPTER_DIR}/")
for nombre in sorted(os.listdir(ADAPTER_DIR)):
    ruta = os.path.join(ADAPTER_DIR, nombre)
    if os.path.isfile(ruta):
        print(f"  {nombre:32s} {os.path.getsize(ruta) / 1e6:8.1f} MB")


def ollama_vivo() -> bool:
    """True si hay un servidor de Ollama contestando en el puerto 11434."""
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except Exception:
        return False


# ¿Hay alguien escuchando en el puerto donde vivirá nuestro modelo? Todavía no.
# Por eso ninguna app puede usar este adapter hoy.
print()
print("¿Hay un servidor escuchando en el puerto 11434?", ollama_vivo())

### 🗺️ El mapa: del adapter al endpoint

<!-- DIAGRAM: ollama-servido. graph LR. Adapter LoRA (MB) pasa por convertir a GGUF (llama.cpp); junto con el modelo base qwen2.5 1.5b cuantizado entran a Ollama en la T4. Ollama sale a un endpoint compatible con OpenAI en localhost 11434 v1, que usan Tu app y Tu agente (Lab 3). Nota: nada sale de la maquina. ASCII, maximo 8 nodos. -->

Antes de escribir código, veamos el camino completo.

In [ ]:
# Del archivo al servicio, en un dibujo.
mostrar_diagrama("ollama-servido")

Léelo de izquierda a derecha:

- **Ollama** es un servidor de modelos: los descarga, los carga en la GPU y los expone en una
  URL. Piensa en él como "Docker para modelos de lenguaje".
- **GGUF** es el formato de archivo que usa Ollama por dentro (viene de llama.cpp). Nuestro
  adapter está en el formato de Hugging Face (safetensors), así que lo vamos a convertir.
- Ollama ya tiene el modelo base `qwen2.5:1.5b`, el mismo Qwen2.5 1.5B Instruct con el que
  entrenamos, pero **cuantizado a 4 bits** para que ocupe poco. Nuestro adapter se aplica
  encima de ese base.
- La salida es un endpoint **compatible con OpenAI**. Cualquier código que hoy le habla a
  OpenAI le puede hablar a nuestro modelo cambiando una sola línea: la URL.

In [ ]:
# ===========================================================================
# LIBERAR MEMORIA DE LA GPU (PARA CASA)
# ===========================================================================
# El entrenamiento dejó cosas en la GPU que ya no necesitamos: el trainer, sus
# gradientes y el estado del optimizador. Ollama va a necesitar su propio espacio en
# la misma T4, así que soltamos lo que sobra.
#
# OJO: NO borramos `model`. Más adelante es el plan B si Ollama no contesta, y volver
# a cargarlo en una T4 es arriesgarse a quedarse sin memoria.
import gc

print(f"Memoria de GPU en uso antes:   {torch.cuda.memory_allocated() / 1e9:.1f} GB")

if "trainer" in globals():
    del trainer          # el trainer guarda referencias al optimizador y a los gradientes
gc.collect()             # Python suelta los objetos que quedaron sin dueño
torch.cuda.empty_cache() # y PyTorch le devuelve a la GPU la memoria que tenía reservada

print(f"Memoria de GPU en uso después: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

### 📦 Instalamos Ollama (una versión fija, y te contamos por qué)

Ollama se instala con un script oficial de una línea. Nosotros le pedimos una versión
concreta, la **0.34.0**, y no la última.

La razón es honesta: a partir de la 0.34.1, Ollama dejó de aceptar adapters LoRA con la
instrucción `ADAPTER`. Las versiones nuevas solo sirven modelos completos. La 0.34.0 es la
última que te deja servir **el adapter chico encima del base**, que es justo lo que queremos
mostrar hoy.

En producción tienes dos caminos si quieres la última versión: fusionar el adapter con el
base y servir el modelo completo, o usar vLLM, que sí sirve adapters LoRA.

In [ ]:
# ===========================================================================
# INSTALAR OLLAMA DENTRO DE COLAB (NARRAR)
# ===========================================================================
# Fijamos la versión. El script oficial acepta la variable OLLAMA_VERSION.
OLLAMA_VERSION = "0.34.0"

# El instalador viene comprimido con zstd. Si la máquina no lo tiene, lo instalamos.
!which zstd > /dev/null || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null)

# El instalador oficial. Tarda unos 15 segundos.
# PARA CASA: si ves "Unable to detect NVIDIA/AMD GPU", tranquilo. Ollama trae sus propias
# librerías de CUDA y usa la T4 igual; lo vas a comprobar con `ollama ps` más abajo.
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh

# El aviso "could not connect to a running Ollama instance" es normal: todavía no
# arrancamos el servidor.
!ollama --version

In [ ]:
# ===========================================================================
# ARRANCAR EL SERVIDOR SIN BLOQUEAR LA CELDA (NARRAR)
# ===========================================================================
import subprocess

OLLAMA_MODELO = "reclamos-conectamx"         # el nombre con el que vamos a crear nuestro modelo
OLLAMA_URL = "http://localhost:11434/v1"     # la ruta compatible con OpenAI de Ollama

# `ollama serve` es un servidor: nunca termina. Si lo corremos con `!`, la celda se
# queda colgada para siempre. Por eso lo lanzamos como proceso aparte, en segundo plano,
# y su salida va a un archivo de log.
#
# Esta celda se puede correr las veces que quieras: si el servidor ya está vivo, no hace nada.
if not ollama_vivo():
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("ollama.log", "w"),
        stderr=subprocess.STDOUT,
        start_new_session=True,  # que no muera si se interrumpe esta celda
        # KEEP_ALIVE=-1: el modelo queda cargado en la GPU. Por defecto Ollama lo descarga
        # a los 5 minutos sin uso, y el Lab 3 pagaría de nuevo el tiempo de carga.
        env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1"},
    )
    # Esperamos a que conteste, como mucho 30 segundos. Normalmente tarda 2.
    for _ in range(60):
        if ollama_vivo():
            break
        time.sleep(0.5)

print("¿Ollama escuchando en el puerto 11434?", ollama_vivo())

In [ ]:
# ===========================================================================
# BAJAR EL MODELO BASE Y "CALENTAR" LA GPU (NARRAR)
# ===========================================================================
# Es el mismo Qwen2.5 1.5B Instruct con el que entrenamos, ya cuantizado a 4 bits por
# Ollama (Q4_K_M). Pesa menos de 1 GB y baja en unos 10 segundos.
!ollama pull qwen2.5:1.5b

# PARA CASA: la primera vez que Ollama usa la GPU en una máquina nueva, CUDA compila sus
# kernels para la T4. Eso tarda más de un minuto, y solo pasa una vez por sesión.
# Para no esperarlo en vivo, mandamos una pregunta mínima en segundo plano ahora mismo.
# Mientras seguimos con las celdas siguientes, la GPU ya se va preparando.
calentar = subprocess.Popen(
    ["curl", "-s", "http://localhost:11434/api/generate", "-d",
     json.dumps({"model": "qwen2.5:1.5b", "prompt": "hola", "stream": False,
                 "options": {"num_predict": 1}})],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("Modelo base listo. La GPU se está calentando en segundo plano.")

### 🧩 Segundo problema: Ollama no entiende nuestro adapter tal como está

Lo natural sería decirle a Ollama: "toma el base y ponle encima la carpeta del adapter".
Probemos exactamente eso y veamos qué pasa.

In [ ]:
# ===========================================================================
# EL INTENTO INGENUO: pasarle la carpeta de Hugging Face tal cual (NARRAR)
# ===========================================================================
# Un Modelfile es la "receta" de un modelo en Ollama: de qué base parte y qué le agregamos.
with open("Modelfile.prueba", "w") as f:
    f.write(f"FROM qwen2.5:1.5b\nADAPTER ./{ADAPTER_DIR}\n")

# Esto va a fallar, y está bien. Con `!` el error se muestra pero el notebook sigue.
!ollama create prueba-safetensors -f Modelfile.prueba

# Lo que pasa por dentro:
# 1. Ollama busca archivos llamados model*.safetensors, y PEFT guarda el nuestro como
#    adapter_model.safetensors. No lo encuentra.
# 2. Y aunque lo encontrara, Ollama solo sabe convertir adapters de Llama y Gemma 2.
#    Para un adapter de Qwen responde "unsupported architecture".
# La salida: convertir nosotros el adapter al formato que Ollama sí entiende, GGUF.

In [ ]:
# ===========================================================================
# CONVERTIR EL ADAPTER A GGUF (NARRAR)
# ===========================================================================
from huggingface_hub import hf_hub_download

# llama.cpp es el proyecto que inventó el formato GGUF, y trae un script que convierte un
# adapter de PEFT a un adapter GGUF. Bajamos solo esa versión del repo (--depth 1), fijada
# en la etiqueta v1.0.0 para que mañana funcione igual que hoy.
!test -d llama.cpp || git clone -q --depth 1 --branch v1.0.0 https://github.com/ggml-org/llama.cpp

ADAPTER_GGUF = "adapter_reclamos.gguf"

# El script lee el adapter y la configuración del modelo base (solo el config.json, no los
# pesos) y escribe un único archivo .gguf. En f16 queda a la mitad del tamaño original.
# No hay que instalar nada: usa torch, transformers y numpy, que Colab ya tiene.
!python llama.cpp/convert_lora_to_gguf.py {ADAPTER_DIR} --outfile {ADAPTER_GGUF} --outtype f16 --base-model-id {MODEL_BASE}

# Plan B automático: si la conversión falló (sin internet a GitHub, un cambio en
# llama.cpp), bajamos el mismo archivo ya convertido del Hub. Nadie se queda trabado.
if not os.path.exists(ADAPTER_GGUF):
    print("La conversión no salió. Bajamos el GGUF ya convertido del Hub.")
    hf_hub_download(ADAPTER_REPO, ADAPTER_GGUF, local_dir=".")

print(f"\n{ADAPTER_GGUF}: {os.path.getsize(ADAPTER_GGUF) / 1e6:.1f} MB")

In [ ]:
# ===========================================================================
# LA RECETA DE NUESTRO MODELO: el Modelfile (NARRAR)
# ===========================================================================
# Cuatro líneas, y cada una importa:
#   FROM      el modelo base, el mismo con el que entrenamos.
#   ADAPTER   nuestro adapter de pocos MB, ahora en GGUF. Esto es lo que aprendió.
#   SYSTEM    EXACTAMENTE el mismo prompt de sistema del entrenamiento. Si cambias una
#             coma, el modelo ve algo que nunca vio al entrenar y el fine-tuning rinde menos.
#   PARAMETER temperature 0: para clasificar queremos siempre la misma respuesta.
MODELFILE = f'''FROM qwen2.5:1.5b
ADAPTER ./{ADAPTER_GGUF}
SYSTEM """{SYSTEM_CLASIFICADOR}"""
PARAMETER temperature 0
'''

with open("Modelfile", "w") as f:
    f.write(MODELFILE)
print(MODELFILE)

# Crear el modelo tarda menos de un segundo: no copia el base, solo registra la receta.
!ollama create {OLLAMA_MODELO} -f Modelfile
!ollama list

# PARA CASA: cargamos nuestro modelo en la GPU en segundo plano, para que la primera
# pregunta de verdad no tenga que esperar la carga.
calentar_nuestro = subprocess.Popen(
    ["curl", "-s", "http://localhost:11434/api/generate", "-d",
     json.dumps({"model": OLLAMA_MODELO, "prompt": "hola", "stream": False,
                 "options": {"num_predict": 1}})],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

Un detalle que se rompe en silencio: el **formato de chat**. Al entrenar, cada reclamo se
armó con la plantilla de Qwen (`<|im_start|>system ... <|im_start|>user ... <|im_start|>assistant`).
El `qwen2.5:1.5b` de Ollama trae esa misma plantilla, y nuestro modelo la hereda del `FROM`.
Lo comprobamos antes del taller: el prompt que arma Ollama y el que armamos al entrenar
producen exactamente la misma respuesta.

Por eso nuestro modelo recibe lo mismo que vio al entrenar: el mismo `SYSTEM`, la misma
plantilla, y el reclamo como mensaje del usuario.

In [ ]:
# ===========================================================================
# EL MISMO CLIENTE DE OPENAI, OTRA URL (NARRAR)
# ===========================================================================
# Fíjate en lo único que cambia respecto a `client`, el de OpenAI:
#   base_url: apunta a NUESTRA máquina, no a internet.
#   api_key:  Ollama no la pide, pero el cliente exige un texto. Cualquiera sirve.
#   timeout y max_retries: si Ollama no contesta, queremos enterarnos rápido, sin reintentos.
cliente_local = OpenAI(base_url=OLLAMA_URL, api_key="ollama", timeout=30, max_retries=0)


def clasificar_local(texto: str) -> str:
    """Clasifica un reclamo con nuestro modelo servido en Ollama.

    Devuelve la respuesta cruda del modelo (un JSON como texto). Si Ollama no contesta,
    usa el mismo modelo que ya está en la memoria de la GPU (plan B).
    """
    try:
        respuesta = cliente_local.chat.completions.create(
            model=OLLAMA_MODELO,
            # Solo mandamos el reclamo. El prompt de sistema ya vive en el Modelfile, así
            # que es imposible que el código y el modelo se desincronicen.
            messages=[{"role": "user", "content": texto}],
            temperature=0,
            max_tokens=64,  # el JSON ocupa unos 25 tokens; esto corta cualquier divagación
        )
        return respuesta.choices[0].message.content.strip()
    except Exception as error:
        # PLAN B: el servidor se cayó o no arrancó. No dejamos al agente sin respuesta:
        # usamos nuestro modelo entrenado, que sigue cargado en la GPU desde el Lab 1.
        # Mismo adapter, mismo prompt de sistema; solo cambia quién lo ejecuta.
        print(f"   (Ollama no contestó: {type(error).__name__}. Usamos el modelo en memoria.)")
        return generar(texto, model)


print("Listo: clasificar_local(texto) definida, con plan B incluido.")

In [ ]:
# ===========================================================================
# LOS MISMOS 5 RECLAMOS, AHORA CONTESTADOS POR NUESTRO SERVIDOR (NARRAR)
# ===========================================================================
# Son los mismos reclamos que vimos fallar con el modelo base en la sección 1.
# PARA CASA: si la primera respuesta tarda unos segundos, es la GPU terminando de cargar
# el modelo. De la segunda en adelante, cada reclamo tarda menos de medio segundo.
for i, texto in enumerate(MUESTRA, 1):
    inicio = time.time()
    respuesta = clasificar_local(texto)
    segundos = time.time() - inicio
    ok = "JSON válido" if parsear_json(respuesta) is not None else "NO es JSON válido"
    print(f"{i}. ({segundos:.1f} s, {ok}) {texto[:70]}...")
    print(f"   -> {respuesta}\n")

# ¿Dónde corre? PROCESSOR dice 100% GPU: todo en la T4, nada en la nube.
!ollama ps

### ✅ Lo que acabamos de hacer

- Tomamos un adapter de unos 70 MB, lo convertimos a GGUF (37 MB) y lo servimos **encima**
  del modelo base, en nuestra propia GPU.
- Lo llamamos con el cliente de OpenAI cambiando solo la URL. Cualquier sistema que ya hable
  con OpenAI puede hablar con nuestro modelo.
- El prompt de sistema vive en el Modelfile: entrenamiento y servicio usan el mismo.

Un costo que conviene decir en voz alta: para servirlo, Ollama usa una base **cuantizada** a
4 bits con otro método (GGUF). En nuestra medición, las tres etiquetas juntas bajaron un poco
respecto al modelo en memoria (68% contra 71%). Comprimir para servir cuesta algo de
calidad; lo medimos en la sección 5. <!-- NUMERO v3 -->

**En producción con tráfico real**, el paso siguiente es vLLM: atiende a muchos usuarios a
la vez y carga adapters LoRA directamente.

Ya tenemos un modelo propio que contesta en una URL. En el Lab 3 se lo damos a un agente
como herramienta, y el modelo grande decide cuándo usarlo.

In [ ]:
# EJERCICIO 5 (opcional, para después)
# Compara nuestro modelo contra el modelo base sin adapter, con el MISMO prompt de sistema.
# Usa cliente_local con model="qwen2.5:1.5b" y manda dos mensajes: uno "system" con
# SYSTEM_CLASIFICADOR y uno "user" con el reclamo. Hazlo con los 3 primeros de MUESTRA y
# ponlos lado a lado con clasificar_local.
# Pista: fíjate en la forma del JSON y en las categorías, no solo en si es JSON válido.
# Ojo: cambiar de modelo hace que Ollama recargue; la primera respuesta tarda unos segundos.
# Solución al final del notebook.

# EJERCICIO 6 (opcional, para después)
# Llama a nuestro modelo SIN Python, con curl, para comprobar que es HTTP puro.
# Haz un POST a http://localhost:11434/v1/chat/completions con un JSON que tenga
# "model": "reclamos-conectamx" y "messages": [{"role": "user", "content": "<un reclamo>"}].
# Pista: en Colab, una línea que empieza con ! corre en la terminal. Escribe el nombre del
# modelo a mano: dentro de un JSON, Colab no reemplaza variables.
# Solución al final del notebook.

## 🤖 Sección 4 (Lab 3): El agente usa nuestro modelo como herramienta

Ya tenemos un modelo propio que clasifica reclamos, servido con Ollama, sin costo por
llamada. Ahora viene la pregunta de verdad: **¿quién decide cuándo usarlo?**

A ConectaMX no le llegan solo reclamos. Le llegan preguntas de horario, gente que quiere
contratar, saludos. Nuestro modelo hace UNA cosa muy bien. Si le mandamos todo, va a
clasificar todo, aunque no sea un reclamo.

Vamos a verlo con la bandeja de hoy.

In [ ]:
# ===========================================================================
# LA BANDEJA DE HOY, Y LA TENTACIÓN DE MANDARLO TODO A NUESTRO MODELO (NARRAR)
# ===========================================================================
# Seis mensajes de un día cualquiera en ConectaMX. Cuatro son reclamos. Dos no: una
# pregunta de horario y una venta. Así llega el correo de verdad: mezclado.
# Dos reclamos traen el número de cliente (1041 y 1028). Guárdalos en la cabeza: en la
# tabla de clientes, el 1041 tiene un cobro duplicado y el 1028 lleva 5 días sin servicio.
BANDEJA = [
    "Hola, este mes me cobraron $1,798 de mi fibra y mi plan es de $899. "
    "Soy el cliente 1041. Quiero que me regresen lo que me cobraron de más.",
    "Llevo cinco días sin internet ni televisión. Soy el cliente 1028 y ya reporté "
    "la falla dos veces. ¿Cuándo van a venir?",
    "El módem se reinicia solo cada rato y se me corta la videollamada del trabajo. "
    "Ya lo desconecté y lo volví a conectar y sigue igual.",
    "Pago 500 megas y en las pruebas de velocidad no paso de 40. Así no puedo trabajar.",
    "¿A qué hora abre la sucursal de Polanco los sábados?",
    "Hola, me acabo de mudar y quiero contratar el Triple Play. ¿Qué incluye y cuánto cuesta?",
]

# La idea ingenua: ya tenemos un modelo propio, gratis y privado. ¡Mandémosle todo!
# Reusamos clasificar_local de la sección anterior, tal cual.
for i, mensaje in enumerate(BANDEJA, start=1):
    print(f"{i}. {mensaje[:60]}...")
    print(f"   nuestro modelo: {clasificar_local(mensaje)}")

# Fíjate en los dos últimos. Nadie se quejó de nada, y aun así salió una clasificación de
# reclamo. No es un error del modelo: lo entrenamos para hacer UNA cosa, y la hace siempre.
# No sabe decir "esto no es lo mío". Alguien tiene que decidir antes si le toca.
# Y para el reclamo del 1041 tampoco basta: clasificarlo no nos dice si el cobro doble es real.

### 🗺️ El agente como ruteador

<!-- DIAGRAM: agente-ruteo. sequenceDiagram. Participantes: Cliente, Agente (gpt-4o-mini), clasificar_reclamo (nuestro modelo, Ollama), consultar_cliente (codigo, pandas). Tres casos: (1) reclamo con numero de cliente: el agente llama a las dos tools y redacta la respuesta; (2) reclamo sin numero: solo clasificar_reclamo; (3) pregunta de horario: el agente responde solo. Notas: costo por llamada 0 sobre nuestro modelo, 0 tokens sobre el codigo. ASCII, maximo ~10 mensajes. -->

Así se reparte el trabajo. Míralo de izquierda a derecha.

In [ ]:
# Quién lee, quién decide y quién trabaja.
mostrar_diagrama("agente-ruteo")

El agente (gpt-4o-mini) es el único que **lee y decide**. No clasifica y no busca datos:
para eso llama a una herramienta (tool).

- Si es un reclamo, llama a **nuestro modelo**. Sin costo por llamada.
- Si trae número de cliente, llama a **código puro**. Cero tokens, siempre la misma respuesta.
- Si no es un reclamo, contesta **él solo**, sin tools.

Tres caminos, y cada uno usa lo más barato que alcanza.

### 🧰 Dos tools: una es código, la otra es nuestro modelo

Una tool es una función de Python que el agente puede pedir que corramos. El agente no ve el
código: ve el **nombre**, el **docstring** y los **argumentos**.

Por eso el docstring no es un comentario cualquiera. Es la instrucción de cuándo usarla.
Fíjate en `clasificar_reclamo`: dice cuándo SÍ y cuándo NO.

In [ ]:
# ===========================================================================
# TOOL 1: CÓDIGO PURO. NI MODELO CHICO NI MODELO GRANDE (NARRAR)
# ===========================================================================
# La tabla de clientes es un CSV con 50 clientes ficticios de ConectaMX.
# En tu empresa esto es tu base de datos o tu CRM.
clientes = pd.read_csv(CLIENTES_URL)


def consultar_cliente(numero_cliente: int) -> str:
    """Busca los datos de un cliente de ConectaMX en la base interna.

    Args:
        numero_cliente: el número de cliente, por ejemplo 1042
    """
    # Un filtro de pandas. Cero tokens, cero alucinaciones, siempre la misma respuesta.
    fila = clientes[clientes["numero_cliente"] == numero_cliente]
    if fila.empty:
        # Devolvemos el error como JSON, no lanzamos una excepción: así el agente lo lee y
        # le contesta al cliente, en vez de tronar a la mitad de la bandeja.
        return json.dumps({"error": f"No existe el cliente {numero_cliente}"},
                          ensure_ascii=False)
    # to_json convierte los tipos de pandas a JSON sin pelear con int64.
    return fila.iloc[0].to_json(force_ascii=False)


# Probamos la tool como lo que es: una función de Python común.
print(consultar_cliente(1041))   # el del cobro duplicado
print(consultar_cliente(9999))   # uno que no existe

In [ ]:
# ===========================================================================
# TOOL 2: NUESTRO MODELO, ENVUELTO COMO TOOL (NARRAR)
# ===========================================================================
from langchain.tools import tool


def clasificar_reclamo(texto: str) -> str:
    """Clasifica un reclamo de un cliente de ConectaMX con nuestro modelo propio.

    Úsala SOLO cuando el mensaje es un reclamo o una queja. No la uses para preguntas
    generales, ventas o saludos.

    Args:
        texto: el mensaje del cliente, tal cual llegó
    """
    # Toda la tool es una línea: llamar a nuestro modelo (Ollama, o el plan B en memoria).
    # El try es lo único que agregamos, y es importante: si una tool lanza una excepción,
    # create_agent NO la atrapa y se cae toda la bandeja. Lo probamos antes del taller.
    # Regla general: una tool nunca truena, devuelve el error como dato.
    try:
        return clasificar_local(texto)
    except Exception as e:
        return json.dumps({"error": f"El modelo propio no respondió: {type(e).__name__}"},
                          ensure_ascii=False)


# Esto es lo que ve gpt-4o-mini de cada tool. parse_docstring=True lee el bloque Args: y
# pone la descripción de cada argumento donde el modelo la busca.
# La primera parte del docstring es la regla de ruteo: "Úsala SOLO cuando...".
for funcion in (clasificar_reclamo, consultar_cliente):
    t = tool(funcion, parse_docstring=True)
    print(t.name)
    print("  cuándo usarla:", t.description.replace("\n", " "))
    print("  argumentos:   ", t.args)

In [ ]:
# ===========================================================================
# EL AGENTE: gpt-4o-mini DECIDE, NUESTRAS TOOLS TRABAJAN (NARRAR)
# ===========================================================================
from langchain.agents import create_agent

# El system prompt dice QUIÉN es el agente y la regla de ruteo en palabras.
# La última oración no es decorativa: sin ella, gpt-4o-mini inventó el horario de la
# sucursal de Polanco en nuestras pruebas. El modelo grande también alucina.
SYSTEM_AGENTE = (
    "Eres el asistente de atención a clientes de ConectaMX, una empresa de "
    "telecomunicaciones en México. Lees un mensaje de un cliente y respondes en español "
    "neutro, con tú, en dos o tres oraciones. "
    "Si el mensaje es un reclamo o una queja, primero usa clasificar_reclamo. "
    "Si además el cliente menciona su número de cliente, usa consultar_cliente para "
    "revisar sus datos antes de responder. "
    "Si el mensaje no es un reclamo (horarios, ventas, dudas generales), responde tú "
    "mismo, sin usar herramientas. "
    "Nunca inventes datos que no te dio una herramienta, como horarios o precios: si no "
    "los tienes, dilo y ofrece pasar el caso con un asesor."
)

agente = create_agent(
    model=f"openai:{MODEL_GPT}",            # el modelo grande, por API: el que decide
    tools=[tool(clasificar_reclamo, parse_docstring=True),   # nuestro modelo
           tool(consultar_cliente, parse_docstring=True)],   # código puro
    system_prompt=SYSTEM_AGENTE,
)

# No es un servicio raro: es un grafo con dos nodos que se turnan.
# "model" decide, "tools" ejecuta lo que pidió, y vuelve a "model" hasta que contesta.
print("nodos del grafo:", list(agente.nodes))

### 📬 La bandeja, por el agente

Ahora pasamos los seis mensajes por el agente, uno por uno. Por cada mensaje vamos a imprimir
**qué tools corrieron de verdad**, no lo que el modelo dijo que iba a hacer.

La prueba está en el historial que devuelve el agente: cada tool que corrió deja un mensaje
de tipo `tool` con su nombre. Y cada respuesta de gpt-4o-mini trae los tokens que gastó,
medidos por la API. Con eso sacamos la cuenta al final.

In [ ]:
# ===========================================================================
# procesar_bandeja: un mensaje a la vez, y anotamos QUÉ TOOLS CORRIERON (PARA CASA)
# ===========================================================================
def procesar_bandeja(bandeja: list[str]) -> list[dict]:
    """Pasa cada mensaje por el agente, imprime una línea por mensaje y un resumen."""
    resultados = []
    for i, mensaje in enumerate(bandeja, start=1):
        # Cada mensaje es una conversación nueva. Sin memoria entre clientes, a propósito.
        respuesta = agente.invoke({"messages": [{"role": "user", "content": mensaje}]})
        mensajes = respuesta["messages"]

        # La prueba de lo que pasó de verdad: los ToolMessage. Cada uno es una tool que
        # SÍ corrió, no una que el modelo dijo que iba a correr.
        tools = [m.name for m in mensajes if m.type == "tool"]
        salida_modelo = [m.content for m in mensajes
                         if m.type == "tool" and m.name == "clasificar_reclamo"]

        # Los tokens que gastó gpt-4o-mini en este mensaje: los reporta la API en cada
        # AIMessage. Suele haber dos por reclamo (pide tools, luego contesta) y uno si no.
        entrada = sum(m.usage_metadata["input_tokens"] for m in mensajes
                      if m.type == "ai" and m.usage_metadata)
        salida = sum(m.usage_metadata["output_tokens"] for m in mensajes
                     if m.type == "ai" and m.usage_metadata)

        print(f"{i}. {mensaje[:70]}...")
        print(f"   tools:     {', '.join(tools) if tools else 'ninguna, contestó gpt-4o-mini solo'}")
        if salida_modelo:
            print(f"   nuestro modelo dijo: {salida_modelo[0]}")
        print(f"   respuesta: {mensajes[-1].content}")
        print()
        resultados.append({"mensaje": mensaje, "tools": tools,
                           "respuesta": mensajes[-1].content,
                           "tokens_entrada": entrada, "tokens_salida": salida})

    # El resumen, medido, no estimado.
    reclamos_locales = sum("clasificar_reclamo" in r["tools"] for r in resultados)
    solo_gpt = sum(not r["tools"] for r in resultados)
    entrada = sum(r["tokens_entrada"] for r in resultados)
    salida = sum(r["tokens_salida"] for r in resultados)
    # Precio de lista de gpt-4o-mini: USD 0.15 por millón de tokens de entrada y
    # USD 0.60 por millón de salida (developers.openai.com, noviembre 2026).
    costo = entrada * 0.15 / 1_000_000 + salida * 0.60 / 1_000_000
    print("=" * 70)
    print(f"Reclamos clasificados por nuestro modelo: {reclamos_locales} de {len(resultados)}"
          " (sin costo por llamada, la clasificación corrió en nuestra GPU)")
    print(f"Mensajes que resolvió gpt-4o-mini solo:   {solo_gpt}")
    print(f"gpt-4o-mini orquestó todo: {entrada:,} tokens de entrada, {salida:,} de salida,"
          f" USD {costo:.4f}")
    print(f"A 10,000 mensajes al día: USD {costo / len(resultados) * 10_000:,.2f} al día"
          " solo por decidir y redactar")
    return resultados

In [ ]:
# ===========================================================================
# LA BANDEJA, POR EL AGENTE (NARRAR)
# ===========================================================================
# Tarda unos 20 segundos: seis mensajes, uno detrás de otro.
# Mientras corre, lee la primera línea: el 1041 debería pasar por las DOS tools.
resultados_bandeja = procesar_bandeja(BANDEJA)

### Lo que acabamos de ver

Tres caminos en la misma bandeja, y cada mensaje tomó el más barato que alcanzaba:

| Mensaje | Quién trabajó | Por qué |
|---|---|---|
| Cobro duplicado del 1041 | nuestro modelo + código | clasificar es lenguaje repetitivo; confirmar el cobro es un dato |
| Falla del 1028 | nuestro modelo + código | igual: clasifica y revisa los días sin servicio |
| Módem y velocidad | nuestro modelo | reclamos sin número de cliente |
| Horario y venta | gpt-4o-mini solo | no son reclamos, nuestro modelo no tenía nada que hacer ahí |

Ésta es la regla para llevarte:

> **Si puede ser código, que sea código. Si es lenguaje repetitivo, tu modelo chico. Al
> modelo grande, solo el juicio.**

Una aclaración honesta: en este diseño gpt-4o-mini **lee todos los mensajes**, porque es el
que decide. La clasificación corre en casa, pero el texto sí sale para el ruteo. Si tus datos
no pueden salir nunca, el ruteador también tiene que ser tuyo: un modelo chico o reglas de
código. Y si otro equipo va a usar tu modelo, ponlo detrás de MCP: es la misma tool, con un
protocolo estándar.

In [ ]:
# EJERCICIO 7 (opcional, para después)
# Arma una bandeja nueva con un solo mensaje: "Soy el cliente 1042, ¿me dices cuánto
# pago al mes?" y pásala por procesar_bandeja.
# No es un reclamo, pero trae número de cliente. ¿Qué tools corren?
# Pista: es el tercer camino de la regla. Código, sin nuestro modelo.
# Solución al final del notebook.

# EJERCICIO 8 (opcional, para después)
# Copia clasificar_reclamo con otro nombre y bórrale del docstring el párrafo
# "Úsala SOLO cuando...". Arma otro agente con esa copia y consultar_cliente, y pásale
# los dos últimos mensajes de BANDEJA. ¿Cambia el ruteo?
# Pista: el agente no ve tu código, solo el nombre, el docstring y los argumentos.
# Solución al final del notebook.

# EJERCICIO 9 (opcional, para después)
# Agrega una tercera tool, escalar_a_humano(motivo), que solo imprima el motivo y
# devuelva un JSON con un número de ticket. Dile en el system prompt que la use cuando
# nuestro modelo devuelva urgencia "alta".
# Pista: es código puro, como consultar_cliente. Arma un agente nuevo, no toques agente.
# Solución al final del notebook.

Ya vimos que funciona: el agente usa nuestro modelo cuando conviene, y solo entonces.

Ahora la pregunta de negocio: **¿cuánto cuesta y qué tan bien clasifica nuestro modelo,
contra gpt-4o-mini haciendo lo mismo?** No lo vamos a adivinar. Lo medimos con los 100
reclamos de prueba.

## 💸 Sección 5: ¿Cuándo vale la pena hacer fine-tuning?

En la sección anterior el agente mandó los reclamos a nuestro modelo y contamos cuántos se
quedaron en casa "a $0". Ahora viene la pregunta incómoda, la que te va a hacer tu jefa o tu
director de finanzas:

> **¿Valió la pena? ¿O `gpt-4o-mini` con un buen prompt hacía lo mismo, más barato?**

No vamos a contestar con opiniones. Vamos a medir, aquí, con los mismos 100 reclamos de
prueba que ningún modelo vio al entrenar, el mismo prompt y la misma función `evaluar` de la
Sección 1. Y vamos a leer la tabla en voz alta, diga lo que diga.

In [ ]:
# ===========================================================================
# LA MEDICIÓN: LOS MISMOS RECLAMOS, LOS DOS MODELOS (NARRAR)
# ===========================================================================
# Misma prueba para los dos, para que la comparación sea justa:
#   - mismos reclamos: df_test, los 100 que ningún modelo vio al entrenar
#   - mismo prompt: SYSTEM_CLASIFICADOR (el de gpt va en la llamada; el nuestro ya lo
#     trae adentro, en el SYSTEM del Modelfile de Ollama)
#   - misma vara: evaluar(), la función de la Sección 1
# Lo único que cambia es quién contesta.
#
# Tarda unos 2 o 3 minutos: son 100 llamadas a la API, una por una, y 100 a nuestro
# modelo. Mientras corre, seguimos leyendo las celdas de abajo.

# Si vas tarde, o si lo estás corriendo en vivo junto con todo el salón con la misma
# clave, bájalo a 20 o 30. Los porcentajes se mueven unos puntos; el costo por reclamo y
# la latencia casi nada, porque son promedios por reclamo.
N_EVAL = 100

df_eval = df_test.head(N_EVAL)

# Los tokens de gpt-4o-mini los cuenta la propia API: predecir_gpt() los va sumando en
# USO_GPT (Sección 1). Tomamos una foto antes y restamos después: así contamos SOLO esta
# medición, con los números que te factura OpenAI, sin estimar nada.
uso_antes = dict(USO_GPT)

# Para nuestro modelo no hay factura: contamos sus tokens nosotros. Para eso guardamos
# cada (reclamo, respuesta) mientras evaluar() mide. Envolvemos la función en vez de
# tocarla: la versión envuelta llama a la original y de paso anota.
respuestas_local = []


def anotando(predecir, destino: list):
    """Devuelve una versión de predecir que además guarda (texto, respuesta) en destino."""
    def envuelta(texto: str) -> str:
        respuesta = predecir(texto)
        destino.append((texto, respuesta))
        return respuesta
    return envuelta


print(f"Midiendo {MODEL_GPT} (API) con {len(df_eval)} reclamos...")
metricas_gpt = evaluar(predecir_gpt, df_eval)

print(f"Midiendo nuestro modelo ({OLLAMA_MODELO}, en Ollama) con {len(df_eval)} reclamos...")
metricas_local = evaluar(anotando(clasificar_local, respuestas_local), df_eval)

llamadas_gpt = USO_GPT["llamadas"] - uso_antes["llamadas"]
entrada_gpt = (USO_GPT["tokens_entrada"] - uso_antes["tokens_entrada"]) / llamadas_gpt
salida_gpt = (USO_GPT["tokens_salida"] - uso_antes["tokens_salida"]) / llamadas_gpt

print("Listo.")
print("gpt-4o-mini   :", metricas_gpt)
print("nuestro modelo:", metricas_local)

### 🧠 Mientras mide: qué aprende un modelo cuando le haces fine-tuning

Antes de ver los números, una advertencia honesta, porque es el error más caro que veo en
empresas:

**El fine-tuning le enseña al modelo formato y comportamiento. No le enseña hechos.**

Lo que sí aprendió nuestro modelo con unos 500 ejemplos:

- a contestar SOLO el JSON, siempre con las mismas tres claves
- a usar nuestras seis categorías y no inventar una séptima
- **nuestro criterio**: qué es urgencia "alta" para ConectaMX. Eso no es un hecho del mundo,
  es una regla de tu empresa, y es justo lo que a un prompt le cuesta describir

Lo que NO le enseñas así: el precio del plan de este mes, la política de reembolsos nueva, el
estado de la red en Monterrey. Eso cambia, y meterlo en los pesos del modelo sale mal:

- Un estudio (Ovadia y otros, 2023) comparó las dos cosas: **RAG le ganó al fine-tuning de
  forma consistente** para meterle conocimiento a un modelo, tanto conocimiento que ya había
  visto como conocimiento nuevo.
- Otro estudio (Gekhman y otros, 2024) encontró que el modelo aprende los hechos nuevos **más
  lento**, y que cuando por fin los aprende, **alucina más**. Su conclusión: el conocimiento
  viene del preentrenamiento; el fine-tuning le enseña a usarlo mejor.

Entonces: **para hechos, RAG** (buscas el dato y se lo pasas en el prompt). **Para formato y
criterio repetitivo, fine-tuning.** Y muchas veces, las dos cosas juntas.

### 📏 Cómo vamos a leer la tabla

| Columna | Qué significa |
|---|---|
| JSON válido % | cuántas respuestas pudimos leer como JSON con las tres claves (`parsear_json`) |
| Categoría % | cuántas acertaron la categoría |
| Urgencia % | cuántas acertaron la urgencia, la etiqueta más "de criterio" |
| Exacto % | cuántas acertaron las TRES cosas: categoría, urgencia y producto |
| Tokens por reclamo | lo que entra (prompt + reclamo) más lo que sale (el JSON) |
| USD por 1,000 | lo que cuesta clasificar mil reclamos |
| Segundos por reclamo | cuánto tarda cada uno, uno por uno |

Los precios son los oficiales de OpenAI en noviembre de 2026 para `gpt-4o-mini`: **USD 0.15
por millón de tokens de entrada y USD 0.60 por millón de salida**. Los tokens de gpt son los
que reporta la API en cada respuesta; los de nuestro modelo los cuenta su propio tokenizer.

El costo de nuestro modelo por token es cero. Pero ojo con ese cero: en un minuto lo
discutimos.

In [ ]:
# ===========================================================================
# LA TABLA: CALIDAD, TOKENS, DINERO Y TIEMPO (NARRAR)
# ===========================================================================
# Precios oficiales de gpt-4o-mini, USD por millón de tokens (noviembre 2026).
# Si lees esto más adelante, revisa https://developers.openai.com/api/docs/pricing
PRECIO_ENTRADA_GPT = 0.15
PRECIO_SALIDA_GPT = 0.60


def tokens_local(texto: str, respuesta: str) -> tuple[int, int]:
    """Tokens de entrada y salida de nuestro modelo, con SU tokenizer (el de Qwen)."""
    mensajes = [{"role": "system", "content": SYSTEM_CLASIFICADOR},
                {"role": "user", "content": texto}]
    # apply_chat_template arma el prompt exacto que ve Qwen, con su formato de chat.
    # Devuelve un diccionario, por eso pedimos ["input_ids"].
    entrada = len(tokenizer.apply_chat_template(
        mensajes, tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"])
    salida = len(tokenizer(respuesta)["input_ids"])
    return entrada, salida


pares_local = [tokens_local(texto, respuesta) for texto, respuesta in respuestas_local]
entrada_local = sum(p[0] for p in pares_local) / len(pares_local)
salida_local = sum(p[1] for p in pares_local) / len(pares_local)

# El dinero: tokens por precio, llevado a mil reclamos.
costo_1000_gpt = 1000 * (entrada_gpt * PRECIO_ENTRADA_GPT
                         + salida_gpt * PRECIO_SALIDA_GPT) / 1_000_000

tabla = pd.DataFrame(
    {
        "JSON válido %": [metricas_gpt["json_valido"], metricas_local["json_valido"]],
        "Categoría %": [metricas_gpt["categoria"], metricas_local["categoria"]],
        "Urgencia %": [metricas_gpt["urgencia"], metricas_local["urgencia"]],
        "Exacto %": [metricas_gpt["exacto"], metricas_local["exacto"]],
        "Tokens por reclamo": [round(entrada_gpt + salida_gpt),
                               round(entrada_local + salida_local)],
        # Nuestro modelo no cobra por token. El costo de la GPU va aparte, abajo.
        "USD por 1,000": [round(costo_1000_gpt, 4), 0.0],
        "Segundos por reclamo": [metricas_gpt["segundos_por_reclamo"],
                                 metricas_local["segundos_por_reclamo"]],
    },
    index=[f"{MODEL_GPT} (API)", "nuestro modelo (Ollama)"],
)
tabla

### 👀 Lo que dice la tabla

Léela de izquierda a derecha y fíjate en tres cosas:

1. **JSON válido.** Los dos llegan al 100%: `gpt-4o-mini` porque Structured Outputs se lo
   garantiza, y el nuestro porque se lo enseñamos. El formato ya no es la discusión.
2. **Urgencia y exacto.** Aquí se ve el criterio. Cuando preparamos el taller, `gpt-4o-mini`
   acertó la categoría 8 de cada 10 veces, pero la urgencia solo la mitad: marca "alta" casi
   todo, porque no sabe qué es urgente para ConectaMX. Nuestro modelo sí lo aprendió de los
   ejemplos. <!-- NUMERO v3: gpt-4o-mini 79/52/95, exacto 39; nuestro modelo servido 95/76/94, exacto 68 -->
   Mira qué salió hoy en tu corrida.
3. **Dinero.** Centavos de dólar por mil reclamos. Sí, leíste bien.

Si `gpt-4o-mini` le gana a nuestro modelo en alguna columna, se dice en voz alta. Un modelo de
1,500 millones de parámetros entrenado tres minutos en una GPU gratis no le gana en todo a uno
de los mejores modelos chicos del mercado. La pregunta es si le gana en **lo que a ti te
importa**.

Y un detalle honesto más: servido en Ollama, nuestro modelo corre sobre una base comprimida
a 4 bits con otro formato (GGUF), y pierde unos puntos contra la versión en memoria del Lab 1.
Comprimir para servir rápido cuesta algo de calidad. Mídelo siempre en tu caso.

In [ ]:
# ===========================================================================
# PROYECCIÓN: 10,000 RECLAMOS AL DÍA (NARRAR)
# ===========================================================================
# El "$0" de nuestro modelo es una verdad a medias. En Colab la GPU te la regalan para
# experimentar, pero Colab no es un servidor: los límites del plan gratis no están
# publicados, te pueden cortar la sesión y no te garantiza la GPU. En producción vas a
# pagar una GPU por hora, la uses o no.
#
# Precio real de una T4 por hora (noviembre 2026):
#   - Hugging Face Spaces, "Nvidia T4 small", todo incluido: USD 0.40
#   - Hugging Face Inference Endpoints, T4 en AWS: USD 0.50
#   - Google Cloud, solo la GPU (la máquina virtual va aparte): USD 0.35
# Usamos 0.40: el más barato que ya incluye la máquina completa.
PRECIO_T4_HORA = 0.40
DIAS_POR_MES = 30
VOLUMEN_DIARIO = 10_000


def proyectar_costo(costo_por_1000: float, reclamos_por_dia: int = 10_000,
                    costo_fijo_por_hora: float = 0.0) -> dict:
    """Proyecta lo que cuesta clasificar reclamos a un volumen diario.

    Args:
        costo_por_1000: USD por cada mil reclamos (lo variable, lo que cobra la API)
        reclamos_por_dia: cuántos reclamos entran al día
        costo_fijo_por_hora: USD por hora que pagas pase lo que pase (la GPU prendida)

    Returns:
        {"por_dia": USD, "por_mes": USD}
    """
    variable_por_dia = costo_por_1000 * reclamos_por_dia / 1000
    fijo_por_dia = costo_fijo_por_hora * 24          # la GPU cobra las 24 horas
    por_dia = variable_por_dia + fijo_por_dia
    return {"por_dia": round(por_dia, 2), "por_mes": round(por_dia * DIAS_POR_MES, 2)}


costo_gpt = proyectar_costo(costo_1000_gpt, VOLUMEN_DIARIO)
costo_local = proyectar_costo(0.0, VOLUMEN_DIARIO, costo_fijo_por_hora=PRECIO_T4_HORA)

print(f"{VOLUMEN_DIARIO:,} reclamos al día:")
print(f"  {MODEL_GPT + ' (API)':<26} USD {costo_gpt['por_dia']:>8,.2f} al día"
      f"   USD {costo_gpt['por_mes']:>9,.2f} al mes")
print(f"  {'nuestro modelo + una T4':<26} USD {costo_local['por_dia']:>8,.2f} al día"
      f"   USD {costo_local['por_mes']:>9,.2f} al mes")

# ¿A partir de cuántos reclamos al día la GPU sale más barata que la API?
# Cuando lo que pagarías a la API en un día iguala lo que cuesta la GPU un día.
costo_por_reclamo_gpt = costo_1000_gpt / 1000
equilibrio_diario = PRECIO_T4_HORA * 24 / costo_por_reclamo_gpt

# ¿Y cuántos reclamos aguanta la T4 en un día, uno por uno, a la latencia medida?
capacidad_diaria = 86_400 / metricas_local["segundos_por_reclamo"]

print()
print(f"Punto de equilibrio contra la T4: {equilibrio_diario:,.0f} reclamos al día")
print(f"Lo que procesa la T4 uno por uno:  {capacidad_diaria:,.0f} reclamos al día")

### 🧾 La parte honesta: a este volumen, la API es más barata

Lo digo yo antes de que alguien lo pregunte: **con 10,000 reclamos al día, `gpt-4o-mini` te
sale más barato que rentar una GPU.** Cuando lo medimos para preparar el taller fueron menos
de USD 0.50 al día contra USD 9.60 de la T4, y el punto de equilibrio cayó alrededor de
200,000 reclamos al día. <!-- NUMERO v3: aprox, con Structured Outputs; la celda de arriba da el número de hoy -->
O sea: la GPU solo se paga sola si la tienes ocupada casi todo el día.

Entonces, ¿para qué hicimos todo esto? No por el dinero, a este volumen. Por otras cuatro
cosas que la tabla de costos no muestra:

1. **Tu criterio.** El prompt no sabe qué es urgencia "alta" en tu empresa. Tus ejemplos sí.
   Mira las columnas "Urgencia" y "Exacto".
2. **Tus datos no salen.** Reclamos con nombres, números de cliente, montos. Con nuestro modelo
   no viajan a ningún lado. Para un banco, una aseguradora o un hospital, esta sola razón decide.
3. **Nadie te lo apaga.** Los modelos de API se retiran y cambian de comportamiento. OpenAI
   incluso está cerrando su propia plataforma de fine-tuning: ya no acepta usuarios nuevos. Tu
   adapter de unos MB es tuyo para siempre.
4. **La GPU no es para una sola tarea.** La misma T4 puede servir varios adapters chicos, y un
   modelo de 1,500 millones también corre en CPU, más lento, en una máquina que ya pagas.

Y el dinero sí entra en la cuenta cuando el modelo grande es más caro, o el volumen es más
alto. Con `gpt-4o` en lugar de `gpt-4o-mini`, el precio por token es unas 16 veces mayor, y el
equilibrio baja a algo así como 13,000 reclamos al día. Lo calculas en el ejercicio de abajo.

In [ ]:
# EJERCICIO 10 (opcional, para después)
# Repite la proyección como si usaras gpt-4o en vez de gpt-4o-mini.
# Precio oficial de gpt-4o (noviembre 2026): USD 2.50 por millón de tokens de entrada y
# USD 10.00 por millón de salida.
# Usa los mismos promedios entrada_gpt y salida_gpt de la medición (gpt-4o usa el mismo
# tokenizer), calcula su costo por 1,000 y pásaselo a proyectar_costo.
# Después calcula su punto de equilibrio contra la T4.
# Pista: es la misma cuenta de costo_1000_gpt, con otros dos precios.
# Solución al final del notebook.

### 🗺️ La plantilla para llevarte: código, RAG, prompt o fine-tuning

Todo lo que vimos hoy cabe en una secuencia de preguntas. Hazlas en este orden, porque cada
una descarta una opción más barata antes de llegar al fine-tuning.

<!-- DIAGRAM: cuando-finetunear. flowchart LR de decision. Seis rombos en cadena: Se puede con codigo? (si: CODIGO reglas regex SQL), Necesita hechos que cambian? (si: RAG), Tarea estrecha y repetitiva? (no: PROMPT modelo grande), Cientos de ejemplos? (no: PROMPT), Los datos deben quedarse en casa? (si: FINE-TUNING), El prompt da la calidad y el costo que necesitas? (si: PROMPT, no: FINE-TUNING). Cuatro salidas. ASCII sin acentos. -->

In [ ]:
# Las preguntas, en orden. El fine-tuning es la última salida, no la primera.
mostrar_diagrama("cuando-finetunear")

Léelo de izquierda a derecha. Fíjate que el fine-tuning es la **última** salida, no la
primera: llegas ahí solo si la tarea es estrecha, se repite, tienes ejemplos, y además los
datos no pueden salir o el prompt no te da la calidad o el costo que necesitas. Y la primera
pregunta es la de la regla de cierre: **si se puede hacer con código, es código.**

**Checklist para copiar a tu próximo proyecto:**

- [ ] ¿Se puede resolver con código (reglas, expresiones regulares, una consulta SQL)? Entonces es código, sin modelo.
- [ ] ¿La respuesta depende de hechos que cambian (precios, políticas, inventario)? Entonces RAG.
- [ ] ¿La tarea es estrecha y siempre igual (clasificar, extraer, dar formato)?
- [ ] ¿Tienes cientos de ejemplos etiquetados? Entre 200 y 500 buenos suele alcanzar para clasificar.
- [ ] ¿Los datos tienen que quedarse dentro de tu empresa?
- [ ] ¿El volumen es alto? Calcula tu punto de equilibrio con `proyectar_costo`.
- [ ] ¿Mediste el prompt primero, con la misma prueba? Si ya da la calidad que necesitas, no entrenes.

Si marcaste estrecha, repetitiva y con ejemplos, y además privacidad, calidad o volumen:
**fine-tuning.** Si no, empieza por el prompt y junta ejemplos mientras tanto.

In [ ]:
# ===========================================================================
# LA PLANTILLA, EN CÓDIGO (NARRAR)
# ===========================================================================
# El mismo diagrama, como función. Fíjate que la aplicamos a sí misma: una decisión con
# reglas claras no necesita un modelo, necesita un if.
def decidir(se_puede_con_codigo: bool, necesita_hechos_que_cambian: bool,
            estrecha_y_repetitiva: bool, ejemplos_etiquetados: int,
            datos_no_pueden_salir: bool, prompt_alcanza: bool) -> str:
    """Recorre el diagrama de decisión y devuelve qué te conviene hacer."""
    if se_puede_con_codigo:
        return "CÓDIGO: reglas, regex o SQL. Cero modelos."
    if necesita_hechos_que_cambian:
        return "RAG: busca los hechos y dáselos al modelo en el prompt."
    if not estrecha_y_repetitiva:
        return "PROMPT con un modelo grande: la tarea cambia demasiado para entrenarla."
    if ejemplos_etiquetados < 200:
        return "PROMPT por ahora, y empieza a juntar ejemplos etiquetados."
    if datos_no_pueden_salir:
        return "FINE-TUNING: tu modelo, en tu infraestructura."
    if prompt_alcanza:
        return "PROMPT: ya funciona; no arregles lo que no está roto."
    return "FINE-TUNING: el prompt no llega a tu calidad, o tu volumen ya lo paga."


# ConectaMX: clasificar reclamos. No se resuelve con reglas (la gente escribe como quiere),
# no depende de hechos que cambian, es estrecha y repetitiva, tenemos unos 500 ejemplos, y
# los reclamos traen datos de clientes.
print("Clasificar reclamos:       ", decidir(False, False, True, 500, True, False))

# Dos casos más de la misma empresa, para ver las otras salidas:
print("Detectar número de cliente:", decidir(True, False, True, 0, True, True))
print("Contestar '¿cuánto cuesta el plan de 500 megas?':",
      decidir(False, True, True, 0, False, True))

In [ ]:
# EJERCICIO 11 (opcional, para después)
# Piensa en una tarea de tu trabajo que hoy le mandas a un modelo grande.
# Contesta las seis preguntas de decidir() con honestidad y córrela.
# Después cambia UNA respuesta (por ejemplo, datos_no_pueden_salir) y fíjate qué rama
# cambia. Si la respuesta es "CÓDIGO", celebra: te ahorraste un modelo.
# Solución al final del notebook.

## 🎯 Cierre: lo que construimos hoy

Llegamos con dos ideas que queríamos tirar:

1. **"El fine-tuning necesita un clúster y mucho dinero."** Lo hicimos con QLoRA en una T4
   gratis de Colab, en minutos, y nos quedó un adapter de MB, no de GB.
2. **"Toda tarea necesita un modelo gigante detrás de una API."** Para una tarea estrecha y
   repetitiva, nuestro modelo chico la hizo con NUESTRO criterio, sin que un solo reclamo
   saliera de casa para clasificarse. Y medimos honestamente cuándo además te ahorra dinero.

Lo que te llevas en este notebook:

- un fine-tuning con QLoRA que deja un adapter chico, listo para reusar
- ese modelo servido con Ollama, detrás de un endpoint compatible con OpenAI
- un agente que lo usa como tool solo cuando conviene
- una tabla de costo y calidad que puedes repetir con tus datos, y una plantilla para decidir

La regla de cierre: **si se puede hacer con código, hazlo con código. Si es lenguaje
repetitivo, usa tu modelo chico. Al modelo grande mándale solo lo que necesita criterio.**

> **Tu modelo, tus datos, tu GPU gratis. El modelo grande decide, el tuyo trabaja.**

### 🎓 Si esto le sirve a tu equipo

Lo que hicimos hoy en una hora lo armamos completo para empresas: fine-tuning, agentes, LLMs
propios, MLOps y nube, con el stack, los datos y los casos de tu equipo, y con el tiempo
suficiente para que quede.

Escanea el código y platicamos: **[www.datatrainersllc.com](https://www.datatrainersllc.com)**

In [ ]:
# QR a Data Trainers, con el mismo ayudante de la primera celda del notebook.
# Si entraste directo aquí y saltaste esa celda, lo volvemos a armar para que esta celda
# funcione sola.
try:
    hacer_qr
except NameError:
    !pip install -q "qrcode[pil]"
    import qrcode
    from qrcode.constants import ERROR_CORRECT_H

    def hacer_qr(url: str, box: int = 6):
        """Arma un QR nítido y fácil de escanear desde lejos."""
        qr = qrcode.QRCode(error_correction=ERROR_CORRECT_H, box_size=box, border=3)
        qr.add_data(url)
        qr.make(fit=True)
        return qr.make_image(fill_color="black", back_color="white")

DATATRAINERS_URL = "https://www.datatrainersllc.com"

# box más grande: la URL es corta, el QR sale con menos módulos, y así queda del mismo
# tamaño que el del principio y se escanea desde el fondo del salón.
display(hacer_qr(DATATRAINERS_URL, box=10))
print(DATATRAINERS_URL)
print()
print("Gracias por venir. Cualquier duda, escríbeme.")

## 🙋 Preguntas

Es tu turno. Algunas que suelen salir, por si quieres empezar por ahí:

- ¿Y con un modelo de 7B en vez de 1.5B?
- ¿Cómo lo pongo en producción con tráfico real? (Spoiler: vLLM.)
- ¿Cada cuánto tengo que reentrenar el adapter?
- ¿Y si mi tarea necesita hechos Y formato? (Spoiler: RAG más fine-tuning, juntos.)

### Para seguir

- El repo del taller, con el dataset y el script que lo generó:
  [github.com/axel-sirota/nerdearla_finetuning_agent](https://github.com/axel-sirota/nerdearla_finetuning_agent)
- El adapter ya entrenado, listo para usar:
  [huggingface.co/axelsirota/qwen2.5-1.5b-reclamos-conectamx-lora](https://huggingface.co/axelsirota/qwen2.5-1.5b-reclamos-conectamx-lora)
- Las soluciones de los ejercicios, justo abajo.

## Soluciones de los mini ejercicios

Todas las soluciones, en orden. Nada de esto hace falta para seguir el taller, y ninguna celda
de arriba depende de estas.

### EJERCICIO 1: ¿y si solo mejoramos el prompt?

Una regla y un ejemplo en el prompt, sobre el mismo modelo base. Spoiler: un modelo de 1.5B
sigue mal las instrucciones largas, y tus reglas reales son decenas. Para eso está el
fine-tuning: los ejemplos quedan en los pesos.

Ojo: corre esta solución antes del Lab 1 si quieres ver al modelo **base**. Después del Lab 1,
`model` ya tiene el adapter, y vas a estar probando al modelo entrenado.

In [ ]:
# SOLUCIÓN EJERCICIO 1: un prompt con una regla y un ejemplo.
# No reemplaza a SYSTEM_CLASIFICADOR: es una variable nueva, solo para esta prueba.
prompt_con_ejemplo = (
    SYSTEM_CLASIFICADOR
    + ' Un modem o router que falla es "equipo". Ejemplo: "mi router se apaga solo" -> '
    + '{"categoria": "equipo", "urgencia": "media", "producto": "fibra"}'
)


def generar_con_prompt(texto: str, prompt: str) -> str:
    """Igual que generar(), pero con el prompt de sistema que le pases."""
    mensajes = [{"role": "system", "content": prompt}, {"role": "user", "content": texto}]
    entrada = tokenizer.apply_chat_template(
        mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        salida = model.generate(**entrada, max_new_tokens=60, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(salida[0][entrada["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()


for texto in MUESTRA:
    print(generar_con_prompt(texto, prompt_con_ejemplo))
# Fíjate: aun con la regla escrita, el modelo base de 1.5B no la aplica bien. El router
# puede seguir saliendo en otra categoría, y a veces inventa productos como "router".
# Un modelo chico sigue mal las instrucciones largas. Y tus reglas reales son decenas:
# meterlas todas en cada llamada sería lento y caro.

### EJERCICIO 2: ¿qué tan seguido es "alta" de verdad?

Pura pandas. Un modelo que dice "alta" a todo acierta solo los que de verdad son "alta": por
eso su urgencia queda en ese porcentaje. No midió nada, solo repitió la misma respuesta.

In [ ]:
# SOLUCIÓN EJERCICIO 2
print("En entrenamiento:")
print(df_train["urgencia"].value_counts(normalize=True).round(2))
print("\nEn los 100 de prueba:")
print(df_test["urgencia"].value_counts(normalize=True).round(2))
# Las tres urgencias están repartidas casi en tercios. Un modelo que dice "alta" a todo
# acierta solo la fracción que de verdad es "alta" en la prueba: cerca de 4 de cada 10.

### EJERCICIO 3: parámetros de LoRA, a mano

Cada matriz A mide `entrada x r` y cada B mide `r x salida`: por capa, `r * (entrada + salida)`.
Con `r=8` entrenas la mitad que con `r=16`.

In [ ]:
# SOLUCIÓN EJERCICIO 3: parámetros de LoRA por aritmética.
capas = {
    "q_proj": (1536, 1536), "k_proj": (1536, 256), "v_proj": (1536, 256),
    "o_proj": (1536, 1536), "gate_proj": (1536, 8960), "up_proj": (1536, 8960),
    "down_proj": (8960, 1536),
}
BLOQUES = 28


def parametros_lora(r: int) -> int:
    """Parámetros entrenables de LoRA en las 7 proyecciones de los 28 bloques."""
    return sum(r * (entrada + salida) for entrada, salida in capas.values()) * BLOQUES


for r in (8, 16, 32):
    print(f"r={r:2d}: {parametros_lora(r):>12,} parámetros "
          f"({100 * parametros_lora(r) / PARAMETROS_BASE:.2f}% del modelo)")
# r=16 da 18,464,768, igual que print_trainable_parameters(). r=8 da la mitad.

### EJERCICIO 4: los errores de urgencia, para leerlos con calma

Solo inferencia con el `model` entrenado. Al leerlos vas a ver que muchos son discutibles: la
urgencia es la etiqueta más "de criterio", y por eso la calidad de tus etiquetas es el techo
de tu modelo.

In [ ]:
# SOLUCIÓN EJERCICIO 4: los errores de urgencia en los primeros 30 reclamos de prueba.
fallos = []
for _, fila in df_test.head(30).iterrows():
    prediccion = parsear_json(generar(fila["texto"], model)) or {}
    if prediccion.get("urgencia") != fila["urgencia"]:
        fallos.append({"texto": fila["texto"][:100],
                       "etiqueta": fila["urgencia"],
                       "modelo": prediccion.get("urgencia")})

print(f"Errores de urgencia en 30 reclamos: {len(fallos)}\n")
for f in fallos:
    print(f"etiqueta={f['etiqueta']:6s} modelo={str(f['modelo']):6s} | {f['texto']}")

### EJERCICIO 5: nuestro modelo contra el base, servidos los dos

Mismo servidor, mismo prompt de sistema, dos modelos. El base responde JSON con otra forma y
se equivoca de categoría; el nuestro responde en una línea, con las etiquetas que le
enseñamos.

In [ ]:
# SOLUCIÓN EJERCICIO 5: nuestro modelo contra el base, mismo prompt de sistema.
for texto in MUESTRA[:3]:
    base = cliente_local.chat.completions.create(
        model="qwen2.5:1.5b",
        messages=[{"role": "system", "content": SYSTEM_CLASIFICADOR},
                  {"role": "user", "content": texto}],
        temperature=0,
        max_tokens=64,
    ).choices[0].message.content.strip()
    print(texto[:70], "...")
    print("   base:   ", base.replace("\n", " "))
    print("   nuestro:", clasificar_local(texto), "\n")

### EJERCICIO 6: el mismo endpoint, con curl

La respuesta es el mismo JSON que devuelve la API de OpenAI: `choices[0].message.content`
trae la clasificación. Cualquier lenguaje que haga un POST puede usar nuestro modelo.

In [ ]:
# SOLUCIÓN EJERCICIO 6: el mismo endpoint, con curl.
# Escribimos el nombre del modelo a mano: cuando el comando tiene llaves de JSON, Colab no
# puede reemplazar {OLLAMA_MODELO} y lo mandaría tal cual.
!curl -s http://localhost:11434/v1/chat/completions -H "Content-Type: application/json" -d '{"model": "reclamos-conectamx", "messages": [{"role": "user", "content": "Me cobraron dos veces el plan de fibra este mes"}]}'

### EJERCICIO 7: número de cliente, pero no es un reclamo

El tercer camino de la regla: el agente necesita un dato, y el dato lo da el código. Nuestro
modelo no se toca, porque nadie se quejó.

In [ ]:
# SOLUCIÓN EJERCICIO 7
resultado = procesar_bandeja(["Soy el cliente 1042, ¿me dices cuánto pago al mes?"])
# Corre solo consultar_cliente, y la respuesta cita el monto de la tabla.

### EJERCICIO 8: ¿y si borramos la regla del docstring?

El agente rutea con lo que lee: nombre, docstring, argumentos y prompt. En nuestras pruebas,
sin la regla escrita el ruteo **no cambió**: con dos tools tan distintas, el nombre
"clasificar" y la palabra "reclamo" le bastan a gpt-4o-mini. La regla escrita es un cinturón
de seguridad: importa cuando las tools se parecen entre sí, o cuando el que rutea es un
modelo más chico.

In [ ]:
# SOLUCIÓN EJERCICIO 8
def clasificar_sin_regla(texto: str) -> str:
    """Clasifica un reclamo de un cliente de ConectaMX con nuestro modelo propio.

    Args:
        texto: el mensaje del cliente, tal cual llegó
    """
    return clasificar_local(texto)


# Un prompt mínimo, SIN la regla de ruteo: así el agente solo tiene el nombre y el docstring.
agente_sin_regla = create_agent(
    model=f"openai:{MODEL_GPT}",
    tools=[tool(clasificar_sin_regla, parse_docstring=True),
           tool(consultar_cliente, parse_docstring=True)],
    system_prompt="Eres el asistente de atención a clientes de ConectaMX. Responde corto.",
)
for mensaje in BANDEJA[-2:]:
    r = agente_sin_regla.invoke({"messages": [{"role": "user", "content": mensaje}]})
    print([m.name for m in r["messages"] if m.type == "tool"], "<-", mensaje[:50])

### EJERCICIO 9: una tercera tool para escalar a una persona

Una capacidad nueva es una función más y una línea más en `tools=`. El CUÁNDO queda en
palabras (docstring y prompt); la acción es código puro. El agente necesita primero la
clasificación, así que `escalar_a_humano` aparece en un segundo turno de tools.

In [ ]:
# SOLUCIÓN EJERCICIO 9
def escalar_a_humano(motivo: str) -> str:
    """Pasa el caso a un asesor humano. Úsala cuando la urgencia del reclamo es alta.

    Args:
        motivo: una oración con el motivo para escalar
    """
    print(f"   [escalado] {motivo}")
    return json.dumps({"ticket": 4815, "estado": "escalado"}, ensure_ascii=False)


agente_con_escalado = create_agent(
    model=f"openai:{MODEL_GPT}",
    tools=[tool(clasificar_reclamo, parse_docstring=True),
           tool(consultar_cliente, parse_docstring=True),
           tool(escalar_a_humano, parse_docstring=True)],
    system_prompt=SYSTEM_AGENTE + " Si clasificar_reclamo devuelve urgencia alta, "
                                  "usa también escalar_a_humano.",
)
r = agente_con_escalado.invoke({"messages": [{"role": "user", "content": BANDEJA[0]}]})
print([m.name for m in r["messages"] if m.type == "tool"])
print(r["messages"][-1].content)

### EJERCICIO 10: la misma cuenta con gpt-4o

El costo por reclamo es tokens por precio, y los tokens casi no cambian entre `gpt-4o-mini`
y `gpt-4o` porque comparten tokenizer. Con un precio unas 16 veces mayor, el punto de
equilibrio contra la T4 baja de cientos de miles a algo así como 13,000 reclamos al día.

In [ ]:
# SOLUCIÓN EJERCICIO 10: la proyección con gpt-4o en vez de gpt-4o-mini.
PRECIO_ENTRADA_GPT4O = 2.50   # USD por millón de tokens de entrada (noviembre 2026)
PRECIO_SALIDA_GPT4O = 10.00   # USD por millón de tokens de salida

# Mismos promedios de tokens de la medición de la Sección 5.
costo_1000_gpt4o = 1000 * (entrada_gpt * PRECIO_ENTRADA_GPT4O
                           + salida_gpt * PRECIO_SALIDA_GPT4O) / 1_000_000

print("gpt-4o, por 1,000 reclamos: USD", round(costo_1000_gpt4o, 4))
print("gpt-4o a 10,000 al día:", proyectar_costo(costo_1000_gpt4o, VOLUMEN_DIARIO))
print("nuestro modelo + T4:   ", proyectar_costo(0.0, VOLUMEN_DIARIO, PRECIO_T4_HORA))

equilibrio_gpt4o = PRECIO_T4_HORA * 24 / (costo_1000_gpt4o / 1000)
print(f"Punto de equilibrio contra la T4: {equilibrio_gpt4o:,.0f} reclamos al día")

### EJERCICIO 11: `decidir()` con una tarea tuya

`decidir` revisa las preguntas en orden y se detiene en la primera que decide. Por eso
cambiar una sola respuesta puede mandarte de fine-tuning a prompt. Un ejemplo resuelto:
resumir llamadas de soporte para el CRM.

In [ ]:
# SOLUCIÓN EJERCICIO 11: un ejemplo resuelto. Resumir llamadas de soporte para el CRM.
# No se hace con reglas, no depende de hechos que cambian, es repetitiva, hay 800
# resúmenes escritos por agentes, y las llamadas tienen datos personales.
print(decidir(se_puede_con_codigo=False, necesita_hechos_que_cambian=False,
              estrecha_y_repetitiva=True, ejemplos_etiquetados=800,
              datos_no_pueden_salir=True, prompt_alcanza=False))

# Cambiamos UNA respuesta: los datos SÍ pueden salir y el prompt ya alcanza.
print(decidir(False, False, True, 800, False, True))